Le but de l’étape 03 est simple :

\[
M_{T0}: Profil + T0 \rightarrow T2
\]

contre

\[
M_{T1}: Profil + T0 + T1 \rightarrow T2
\]

L’objectif est de vérifier :

> **Est-ce que l’ajout de T1 améliore la prédiction de la performance finale T2 ?**

Donc l’étape 03 sert à :

1. construire les deux modèles ;
2. les comparer sur les **mêmes apprenants** ;
3. mesurer MAE, RMSE et R² ;
4. identifier le meilleur algorithme ;
5. retenir le meilleur framework pour la suite.

Votre résultat principal est :

\[
\boxed{M_{T1} \text{ est meilleur que } M_{T0}}
\]

Donc **T1 apporte une information prédictive utile pour prévoir T2**.

# 03 — Modélisation prédictive M_T0 vs M_T1

## Objectif

Comparer deux cadres de prédiction de la performance académique finale T2 :

\[
M_{T0}: Profil + T0 \rightarrow T2
\]

\[
M_{T1}: Profil + T0 + T1 \rightarrow T2
\]

La comparaison principale est réalisée sur les mêmes 125 apprenants afin
d'évaluer l'apport informationnel de T1 sans confondre cet effet avec un
changement d'échantillon.

## Principes méthodologiques

- M_T0 complet : N = 128
- M_T1 complet : N = 125
- comparaison principale M_T0 vs M_T1 : N = 125
- mêmes apprenants ;
- mêmes folds externes ;
- T2 n'est jamais utilisé comme prédicteur ;
- les prétraitements seront réalisés à l'intérieur des folds ;
- la cible principale sera le score T2 continu ;
- aucun seuil Débutant / Intermédiaire / Avancé n'est inventé ;
- les corrigés T1 et T2 doivent être validés avant le scoring ;
- T1 et T2 comprennent 15 questions propres à chaque parcours DA, BI, DS et IA.

La modélisation principale est donc d'abord formulée comme un problème
de régression.

# Cellule 1 — Imports

In [1]:
import os
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"Found unknown categories in columns.*",
    category=UserWarning
)

os.environ["PYTHONWARNINGS"] = (
    "ignore:Found unknown categories in columns:UserWarning"
)

In [2]:
# ============================================================
# CELLULE 1 — Imports et configuration
# ============================================================

from pathlib import Path

import json
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


pd.set_option(
    "display.max_columns",
    250
)

pd.set_option(
    "display.max_rows",
    150
)

pd.set_option(
    "display.width",
    240
)


RANDOM_STATE = 42

OUTER_FOLDS = 5
INNER_FOLDS = 4


print(
    "✅ Environnement Notebook 03 prêt"
)

✅ Environnement Notebook 03 prêt


# Cellule 2 — Répertoires

In [3]:
# ============================================================
# CELLULE 2 — Répertoires du projet
# ============================================================

def trouver_racine_projet():

    candidats = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent
    ]


    for racine in candidats:

        if (
            racine
            / "outputs"
            / "01_nettoyage"
        ).exists() and (
            racine
            / "outputs"
            / "02_diagnostic_T0"
        ).exists():

            return racine.resolve()


    raise FileNotFoundError(
        "Impossible de trouver les sorties "
        "des notebooks 01 et 02."
    )


PROJECT_ROOT = trouver_racine_projet()


INPUT_01 = (
    PROJECT_ROOT
    / "outputs"
    / "01_nettoyage"
)


INPUT_02 = (
    PROJECT_ROOT
    / "outputs"
    / "02_diagnostic_T0"
)


OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "03_modelisation_MT0_MT1"
)


FIGURE_DIR = (
    OUTPUT_DIR
    / "figures"
)


TABLE_DIR = (
    OUTPUT_DIR
    / "tables"
)


MODEL_DIR = (
    OUTPUT_DIR
    / "models"
)


for dossier in [
    OUTPUT_DIR,
    FIGURE_DIR,
    TABLE_DIR,
    MODEL_DIR
]:

    dossier.mkdir(
        parents=True,
        exist_ok=True
    )


print(
    "PROJECT_ROOT :",
    PROJECT_ROOT
)

print(
    "INPUT 01     :",
    INPUT_01
)

print(
    "INPUT 02     :",
    INPUT_02
)

print(
    "OUTPUT       :",
    OUTPUT_DIR
)

PROJECT_ROOT : /home/andrycomputer/Documents/Model-eval-pers-elearning
INPUT 01     : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/01_nettoyage
INPUT 02     : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/02_diagnostic_T0
OUTPUT       : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/03_modelisation_MT0_MT1


# Cellule 3 — Charger les données validées

In [4]:
# ============================================================
# CELLULE 3 — Chargement
# ============================================================

FILES = {

    "D1":
        INPUT_01
        / "D1_profils_nettoyes.csv",

    "T1":
        INPUT_01
        / "T1_nettoye.csv",

    "T2":
        INPUT_01
        / "T2_nettoye.csv",

    "T1_LONG":
        INPUT_01
        / "T1_reponses_long.csv",

    "T2_LONG":
        INPUT_01
        / "T2_reponses_long.csv",

    "T0_SCORES":
        INPUT_02
        / "T0_scores_domaines.csv",

    "DIAGNOSTIC_T0":
        INPUT_02
        / "diagnostic_initial_T0.csv"
}


for nom, chemin in FILES.items():

    if not chemin.exists():

        raise FileNotFoundError(
            f"{nom} absent : {chemin}"
        )


D1 = pd.read_csv(
    FILES["D1"]
)


T1 = pd.read_csv(
    FILES["T1"]
)


T2 = pd.read_csv(
    FILES["T2"]
)


T1_LONG = pd.read_csv(
    FILES["T1_LONG"]
)


T2_LONG = pd.read_csv(
    FILES["T2_LONG"]
)


T0_SCORES = pd.read_csv(
    FILES["T0_SCORES"]
)


DIAGNOSTIC_T0 = pd.read_csv(
    FILES["DIAGNOSTIC_T0"]
)


print(
    "D1              :",
    D1.shape
)

print(
    "T0 scores       :",
    T0_SCORES.shape
)

print(
    "Diagnostic T0   :",
    DIAGNOSTIC_T0.shape
)

print(
    "T1              :",
    T1.shape
)

print(
    "T2              :",
    T2.shape
)

print(
    "T1 réponses long:",
    T1_LONG.shape
)

print(
    "T2 réponses long:",
    T2_LONG.shape
)

D1              : (472, 22)
T0 scores       : (305, 5)
Diagnostic T0   : (305, 24)
T1              : (196, 77)
T2              : (206, 75)
T1 réponses long: (3150, 8)
T2 réponses long: (3090, 8)


# Cellule 4 — Revalider les cohortes

In [5]:
# ============================================================
# CELLULE 4 — Vérification des identifiants et cohortes
# ============================================================

ID = "_ID_NORM"


for nom, df in [

    ("D1", D1),

    ("T0", T0_SCORES),

    ("T1", T1),

    ("T2", T2)

]:

    if ID not in df.columns:

        raise ValueError(
            f"{ID} absent de {nom}"
        )


    if (
        df[ID]
        .dropna()
        .duplicated()
        .any()
    ):

        raise ValueError(
            f"IDs dupliqués dans {nom}"
        )


IDS_D1 = set(
    D1[ID]
    .dropna()
)


IDS_T0 = set(
    T0_SCORES[ID]
    .dropna()
)


IDS_T1 = set(
    T1[ID]
    .dropna()
)


IDS_T2 = set(
    T2[ID]
    .dropna()
)


COHORTE_MT0 = (
    IDS_D1
    & IDS_T0
    & IDS_T2
)


COHORTE_MT1 = (
    IDS_D1
    & IDS_T0
    & IDS_T1
    & IDS_T2
)


COHORTE_COMPARAISON = (
    set(
        COHORTE_MT1
    )
)


print(
    "D1 :",
    len(IDS_D1)
)

print(
    "T0 :",
    len(IDS_T0)
)

print(
    "T1 :",
    len(IDS_T1)
)

print(
    "T2 :",
    len(IDS_T2)
)


print()


print(
    "M_T0 :",
    len(COHORTE_MT0)
)

print(
    "M_T1 :",
    len(COHORTE_MT1)
)

print(
    "Comparaison :",
    len(
        COHORTE_COMPARAISON
    )
)


assert len(COHORTE_MT0) == 128
assert len(COHORTE_MT1) == 125
assert len(COHORTE_COMPARAISON) == 125


print(
    "\n✅ Cohortes conformes aux notebooks précédents."
)

D1 : 472
T0 : 305
T1 : 196
T2 : 206

M_T0 : 128
M_T1 : 125
Comparaison : 125

✅ Cohortes conformes aux notebooks précédents.


# Cellule 5 — Fonctions de traitement des QCM

In [6]:
# ============================================================
# CELLULE 5 — Fonctions pour T1/T2
# ============================================================

def normaliser_texte(
    valeur
):

    if pd.isna(valeur):

        return pd.NA


    texte = re.sub(
        r"\s+",
        " ",
        str(valeur)
    ).strip()


    if texte.upper() in {

        "",
        "NAN",
        "NONE",
        "NULL",
        "<NA>"

    }:

        return pd.NA


    return texte


def extraire_option(
    valeur
):

    """
    Extrait A, B, C, D ou E depuis :

    B
    B.
    B. réponse
    B) réponse
    """

    if pd.isna(valeur):

        return pd.NA


    texte = (
        str(valeur)
        .strip()
        .upper()
    )


    if texte in {
        "A",
        "B",
        "C",
        "D",
        "E"
    }:

        return texte


    match = re.match(

        r"^\s*([A-E])"
        r"(?:\s*[\.\)\-:]|\s+)",

        texte

    )


    if match:

        return match.group(1)


    return pd.NA


print(
    "✅ Fonctions QCM prêtes"
)

✅ Fonctions QCM prêtes


# Cellule 6 — Vérifier T1 et T2

In [7]:
# ============================================================
# CELLULE 6 — Structure T1 / T2
# ============================================================

colonnes_obligatoires = {

    ID,
    "PARCOURS",
    "QUESTION_INDEX",
    "QUESTION",
    "REPONSE"

}


for nom, df in [

    ("T1", T1_LONG),

    ("T2", T2_LONG)

]:

    manque = (
        colonnes_obligatoires
        - set(df.columns)
    )


    if manque:

        raise ValueError(

            f"{nom} : colonnes absentes : "
            f"{manque}"

        )


structure_t1 = (

    T1_LONG

    .groupby(
        "PARCOURS"
    )

    ["QUESTION_INDEX"]

    .nunique()

    .reindex(
        [
            "DA",
            "BI",
            "DS",
            "IA"
        ]
    )

)


structure_t2 = (

    T2_LONG

    .groupby(
        "PARCOURS"
    )

    ["QUESTION_INDEX"]

    .nunique()

    .reindex(
        [
            "DA",
            "BI",
            "DS",
            "IA"
        ]
    )

)


print(
    "T1"
)

display(
    structure_t1
    .rename("N_ITEMS")
    .to_frame()
)


print(
    "T2"
)

display(
    structure_t2
    .rename("N_ITEMS")
    .to_frame()
)


if not (
    structure_t1 == 15
).all():

    raise ValueError(
        "T1 doit avoir 15 items par parcours."
    )


if not (
    structure_t2 == 15
).all():

    raise ValueError(
        "T2 doit avoir 15 items par parcours."
    )


print(
    "✅ 15 questions × 4 parcours pour T1 et T2"
)

T1


,N_ITEMS
PARCOURS,
DA,15
BI,15
DS,15
IA,15


T2


,N_ITEMS
PARCOURS,
DA,15
BI,15
DS,15
IA,15


✅ 15 questions × 4 parcours pour T1 et T2


# Cellule 7 — Construire les corrigés à valider
4\ parcours X times 15= 60 questions 

In [8]:
# ============================================================
# CELLULE 7 — Templates des corrigés T1/T2
# ============================================================

def construire_template_corrige(
    df_long,
    etape
):

    resultat = []


    groupes = df_long.groupby(

        [
            "PARCOURS",
            "QUESTION_INDEX",
            "QUESTION"
        ],

        dropna=False

    )


    for (
        parcours,
        numero,
        question
    ), groupe in groupes:


        options = sorted(

            groupe[
                "REPONSE"
            ]

            .dropna()

            .map(
                extraire_option
            )

            .dropna()

            .unique()

            .tolist()

        )


        modalites = sorted(

            groupe[
                "REPONSE"
            ]

            .dropna()

            .astype(str)

            .unique()

            .tolist()

        )


        resultat.append({

            "ETAPE":
                etape,

            "PARCOURS":
                parcours,

            "QUESTION_INDEX":
                int(numero),

            "QUESTION":
                question,

            "OPTIONS_OBSERVEES":
                " | ".join(
                    options
                ),

            "MODALITES_OBSERVEES":
                " || ".join(
                    modalites
                ),

            "REPONSE_CORRECTE":
                "",

            "COMPETENCE":
                "",

            "JUSTIFICATION_COURTE":
                ""

        })


    return (

        pd.DataFrame(
            resultat
        )

        .sort_values(
            [
                "PARCOURS",
                "QUESTION_INDEX"
            ]
        )

        .reset_index(
            drop=True
        )

    )


template_t1 = (
    construire_template_corrige(
        T1_LONG,
        "T1"
    )
)


template_t2 = (
    construire_template_corrige(
        T2_LONG,
        "T2"
    )
)


print(
    "T1 :",
    len(template_t1),
    "questions"
)


print(
    "T2 :",
    len(template_t2),
    "questions"
)


assert len(template_t1) == 60
assert len(template_t2) == 60


display(
    template_t1.head(20)
)

T1 : 60 questions
T2 : 60 questions


,ETAPE,PARCOURS,QUESTION_INDEX,QUESTION,OPTIONS_OBSERVEES,MODALITES_OBSERVEES,REPONSE_CORRECTE,COMPETENCE,JUSTIFICATION_COURTE
0,T1,BI,1,Quel niveau d'analyse repond a la question : Q...,A | B | C | D,A) Predictif || B) Prescriptif || C) Descripti...,,,
1,T1,BI,2,Lequel de ces ensembles constitue correctement...,A | B | C,A) Power Query uniquement || B) Power BI Deskt...,,,
2,T1,BI,3,Quel mode de connexion Power BI est recommande...,A | B | C | D,A) DirectQuery — lecture en direct sur la sour...,,,
3,T1,BI,4,Un analyste Elite Power BI doit principalement...,A | B | C,A) Que s'est-il passe ? || B) Combien de ligne...,,,
4,T1,BI,5,"Dans Power Query, l'etape Load du modele ETL c...",A | B | C | D,A) La connexion a la source de donnees || B) L...,,,
5,T1,BI,6,Le Fill Down dans Power Query permet de corrig...,A | B | C | D,A) Les valeurs en double sur la cle primaire |...,,,
6,T1,BI,7,Quelle est la methode recommandee pour l'Unpiv...,A | B | C | D,A) Selectionner les colonnes mois puis Depivot...,,,
7,T1,BI,8,"Dans l'architecture 3 couches Power Query, que...",A | B | C | D,A) Src_ — sources brutes (Load desactive) || B...,,,
8,T1,BI,9,Pourquoi faut-il imperativement typer la colon...,A | B | C | D,A) Pour reduire la taille du fichier Power BI ...,,,
9,T1,BI,10,Quel est l'ordre correct des operations lors d...,A | B | C | D,A) Supprimer les lignes vides puis Fill Down |...,,,


# Cellule 8 — Export des corrigés

In [9]:
# ============================================================
# CELLULE 8 — Export des templates
# ============================================================

CORRIGE_T1_TEMPLATE = (

    OUTPUT_DIR
    / "corrige_T1_a_valider.csv"

)


CORRIGE_T2_TEMPLATE = (

    OUTPUT_DIR
    / "corrige_T2_a_valider.csv"

)


CORRIGE_T1_VALIDE = (

    OUTPUT_DIR
    / "corrige_T1_valide.csv"

)


CORRIGE_T2_VALIDE = (

    OUTPUT_DIR
    / "corrige_T2_valide.csv"

)


if not CORRIGE_T1_TEMPLATE.exists():

    template_t1.to_csv(

        CORRIGE_T1_TEMPLATE,

        index=False,

        encoding="utf-8-sig"

    )

    print(
        "✅ Créé :",
        CORRIGE_T1_TEMPLATE
    )

else:

    print(
        "ℹ️ Existe déjà :",
        CORRIGE_T1_TEMPLATE
    )


if not CORRIGE_T2_TEMPLATE.exists():

    template_t2.to_csv(

        CORRIGE_T2_TEMPLATE,

        index=False,

        encoding="utf-8-sig"

    )

    print(
        "✅ Créé :",
        CORRIGE_T2_TEMPLATE
    )

else:

    print(
        "ℹ️ Existe déjà :",
        CORRIGE_T2_TEMPLATE
    )


print(
    "\nT1 à valider :",
    CORRIGE_T1_TEMPLATE
)

print(
    "T2 à valider :",
    CORRIGE_T2_TEMPLATE
)

ℹ️ Existe déjà : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/03_modelisation_MT0_MT1/corrige_T1_a_valider.csv
ℹ️ Existe déjà : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/03_modelisation_MT0_MT1/corrige_T2_a_valider.csv

T1 à valider : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/03_modelisation_MT0_MT1/corrige_T1_a_valider.csv
T2 à valider : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/03_modelisation_MT0_MT1/corrige_T2_a_valider.csv


J’ai contrôlé les deux corrigés fournis. Ils sont techniquement exploitables pour le scoring :
Contrôle	T1	T2
Questions	60	60
DA	15	15
BI	15	15
DS	15	15
IA	15	15
QUESTION_INDEX par parcours	1–15	1–15
Doublons (PARCOURS, QUESTION_INDEX)	0	0
Réponses correctes manquantes	0	0
Compétences manquantes	0	0
Justifications manquantes	0	0
Réponse correcte hors A–E	0	0
Réponse correcte absente des options observées	0	0

In [10]:
# ============================================================
# CELLULE 9 — Validation officielle des corrigés T1 / T2
# ============================================================

def valider_corrige(
    chemin_source,
    chemin_valide,
    etape
):

    corrige = pd.read_csv(
        chemin_source
    )

    colonnes_obligatoires = [
        "ETAPE",
        "PARCOURS",
        "QUESTION_INDEX",
        "QUESTION",
        "OPTIONS_OBSERVEES",
        "REPONSE_CORRECTE",
        "COMPETENCE",
        "JUSTIFICATION_COURTE"
    ]


    absentes = [
        c
        for c in colonnes_obligatoires
        if c not in corrige.columns
    ]


    if absentes:

        raise ValueError(
            f"{etape} — colonnes manquantes : {absentes}"
        )


    # ----------------------------
    # Contrôles généraux
    # ----------------------------

    if len(corrige) != 60:

        raise ValueError(
            f"{etape} doit contenir 60 questions, "
            f"mais {len(corrige)} ont été trouvées."
        )


    corrige["PARCOURS"] = (
        corrige["PARCOURS"]
        .astype(str)
        .str.strip()
        .str.upper()
    )


    corrige["QUESTION_INDEX"] = pd.to_numeric(
        corrige["QUESTION_INDEX"],
        errors="raise"
    ).astype(int)


    corrige["REPONSE_CORRECTE"] = (
        corrige["REPONSE_CORRECTE"]
        .astype(str)
        .str.strip()
        .str.upper()
    )


    if not corrige["REPONSE_CORRECTE"].isin(
        ["A", "B", "C", "D", "E"]
    ).all():

        display(
            corrige[
                ~corrige[
                    "REPONSE_CORRECTE"
                ].isin(
                    ["A", "B", "C", "D", "E"]
                )
            ]
        )

        raise ValueError(
            f"{etape} — réponse correcte invalide."
        )


    # ----------------------------
    # 15 questions par parcours
    # ----------------------------

    controle_parcours = (
        corrige
        .groupby("PARCOURS")
        ["QUESTION_INDEX"]
        .agg(
            N="count",
            MIN="min",
            MAX="max",
            N_UNIQUES="nunique"
        )
        .reindex(
            ["DA", "BI", "DS", "IA"]
        )
    )


    display(
        controle_parcours
    )


    if not (
        (controle_parcours["N"] == 15)
        & (controle_parcours["MIN"] == 1)
        & (controle_parcours["MAX"] == 15)
        & (controle_parcours["N_UNIQUES"] == 15)
    ).all():

        raise ValueError(
            f"{etape} — structure des questions incorrecte."
        )


    # ----------------------------
    # Doublons
    # ----------------------------

    doublons = corrige.duplicated(
        subset=[
            "PARCOURS",
            "QUESTION_INDEX"
        ],
        keep=False
    )


    if doublons.any():

        display(
            corrige[doublons]
        )

        raise ValueError(
            f"{etape} — questions dupliquées."
        )


    # ----------------------------
    # Option correcte observée
    # ----------------------------

    problemes = []


    for _, row in corrige.iterrows():

        options = {
            x.strip().upper()
            for x in str(
                row["OPTIONS_OBSERVEES"]
            ).split("|")
        }


        if (
            row["REPONSE_CORRECTE"]
            not in options
        ):

            problemes.append({
                "PARCOURS":
                    row["PARCOURS"],

                "QUESTION_INDEX":
                    row["QUESTION_INDEX"],

                "REPONSE_CORRECTE":
                    row["REPONSE_CORRECTE"],

                "OPTIONS_OBSERVEES":
                    row["OPTIONS_OBSERVEES"]
            })


    if problemes:

        display(
            pd.DataFrame(
                problemes
            )
        )

        raise ValueError(
            f"{etape} — option correcte "
            "absente des options observées."
        )


    # ----------------------------
    # Sauvegarde version validée
    # ----------------------------

    corrige.to_csv(
        chemin_valide,
        index=False,
        encoding="utf-8-sig"
    )


    print(
        f"✅ {etape} validé :",
        chemin_valide
    )


    return corrige

In [11]:
corrige_t1 = valider_corrige(
    CORRIGE_T1_TEMPLATE,
    CORRIGE_T1_VALIDE,
    "T1"
)


corrige_t2 = valider_corrige(
    CORRIGE_T2_TEMPLATE,
    CORRIGE_T2_VALIDE,
    "T2"
)

,N,MIN,MAX,N_UNIQUES
PARCOURS,,,,
DA,15,1,15,15
BI,15,1,15,15
DS,15,1,15,15
IA,15,1,15,15


✅ T1 validé : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/03_modelisation_MT0_MT1/corrige_T1_valide.csv


,N,MIN,MAX,N_UNIQUES
PARCOURS,,,,
DA,15,1,15,15
BI,15,1,15,15
DS,15,1,15,15
IA,15,1,15,15


✅ T2 validé : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/03_modelisation_MT0_MT1/corrige_T2_valide.csv


# Cellule 10 — Fonction de scoring T1/T2

Pour T1/T2 :
\[
Score = \frac{N_{\text{correct}}}{15}\times100
\]
Nous conserverons également N_CORRECTES, N_REPONSES et le taux de complétude.

In [12]:
# ============================================================
# CELLULE 10 — Fonction robuste de scoring T1 / T2
# ============================================================

def scorer_evaluation(
    df_long,
    corrige,
    etape
):

    donnees = df_long.copy()

    # --------------------------------------------------------
    # 1. Garder uniquement les observations identifiables
    # --------------------------------------------------------

    donnees = donnees[
        donnees[ID].notna()
    ].copy()


    # --------------------------------------------------------
    # 2. Normalisation des clés
    # --------------------------------------------------------

    donnees["PARCOURS"] = (
        donnees["PARCOURS"]
        .astype("string")
        .str.strip()
        .str.upper()
    )


    donnees["QUESTION_INDEX"] = pd.to_numeric(
        donnees["QUESTION_INDEX"],
        errors="raise"
    ).astype(int)


    # --------------------------------------------------------
    # 3. Préparation du corrigé
    # --------------------------------------------------------

    correction = corrige[
        [
            "PARCOURS",
            "QUESTION_INDEX",
            "QUESTION",
            "REPONSE_CORRECTE",
            "COMPETENCE"
        ]
    ].copy()


    correction["PARCOURS"] = (
        correction["PARCOURS"]
        .astype("string")
        .str.strip()
        .str.upper()
    )


    correction["QUESTION_INDEX"] = pd.to_numeric(
        correction["QUESTION_INDEX"],
        errors="raise"
    ).astype(int)


    correction["REPONSE_CORRECTE"] = (
        correction["REPONSE_CORRECTE"]
        .astype("string")
        .str.strip()
        .str.upper()
    )


    correction = correction.rename(
        columns={
            "QUESTION":
                "QUESTION_CORRIGE"
        }
    )


    # --------------------------------------------------------
    # 4. Jointure réponse ↔ corrigé
    # --------------------------------------------------------

    donnees = donnees.merge(

        correction,

        on=[
            "PARCOURS",
            "QUESTION_INDEX"
        ],

        how="left",

        validate="many_to_one"

    )


    # --------------------------------------------------------
    # 5. Vérifier qu'aucun item n'a perdu son corrigé
    # --------------------------------------------------------

    sans_corrige = (
        donnees[
            "REPONSE_CORRECTE"
        ]
        .isna()
    )


    if sans_corrige.any():

        display(
            donnees.loc[
                sans_corrige,
                [
                    "PARCOURS",
                    "QUESTION_INDEX",
                    "QUESTION"
                ]
            ]
            .drop_duplicates()
        )

        raise ValueError(
            f"{etape} — certaines questions "
            "n'ont pas de corrigé associé."
        )


    # --------------------------------------------------------
    # 6. Contrôle du texte des questions
    # --------------------------------------------------------

    question_source = (
        donnees["QUESTION"]
        .astype("string")
        .str.strip()
    )


    question_corrige = (
        donnees[
            "QUESTION_CORRIGE"
        ]
        .astype("string")
        .str.strip()
    )


    mismatch = (

        question_source.notna()

        & question_corrige.notna()

        & question_source.ne(
            question_corrige
        )

    )


    if mismatch.any():

        display(
            donnees.loc[
                mismatch,
                [
                    "PARCOURS",
                    "QUESTION_INDEX",
                    "QUESTION",
                    "QUESTION_CORRIGE"
                ]
            ]
            .drop_duplicates()
            .head(30)
        )

        raise ValueError(
            f"{etape} — incohérence entre "
            "les questions des données et du corrigé."
        )


    # --------------------------------------------------------
    # 7. Identifier les vraies non-réponses
    # --------------------------------------------------------

    donnees[
        "REPONSE_BRUTE_MANQUANTE"
    ] = (
        donnees["REPONSE"]
        .isna()
    )


    # --------------------------------------------------------
    # 8. Extraction A/B/C/D/E
    # --------------------------------------------------------

    donnees[
        "OPTION_REPONSE"
    ] = (
        donnees["REPONSE"]
        .map(
            extraire_option
        )
        .astype("string")
    )


    donnees[
        "OPTION_CORRECTE"
    ] = (
        donnees[
            "REPONSE_CORRECTE"
        ]
        .astype("string")
        .str.strip()
        .str.upper()
    )


    # --------------------------------------------------------
    # 9. Détecter une réponse présente mais non interprétable
    # --------------------------------------------------------

    donnees[
        "OPTION_NON_RECONNUE"
    ] = (

        ~donnees[
            "REPONSE_BRUTE_MANQUANTE"
        ]

        & donnees[
            "OPTION_REPONSE"
        ].isna()

    )


    n_non_reconnues = int(
        donnees[
            "OPTION_NON_RECONNUE"
        ].sum()
    )


    if n_non_reconnues > 0:

        print(
            f"⚠️ {etape} — "
            f"{n_non_reconnues} réponse(s) "
            "présente(s) mais option A-E non reconnue(s)."
        )

        display(
            donnees.loc[
                donnees[
                    "OPTION_NON_RECONNUE"
                ],
                [
                    ID,
                    "PARCOURS",
                    "QUESTION_INDEX",
                    "REPONSE"
                ]
            ].head(30)
        )

        raise ValueError(
            f"{etape} contient des réponses "
            "non interprétables. "
            "Corriger avant le scoring."
        )


    # --------------------------------------------------------
    # 10. Comparaison robuste sans ambiguïté de pd.NA
    # --------------------------------------------------------

    option_reponse = (
        donnees[
            "OPTION_REPONSE"
        ]
        .fillna("__ABSENCE__")
    )


    option_correcte = (
        donnees[
            "OPTION_CORRECTE"
        ]
        .fillna("__CORRIGE_ABSENT__")
    )


    donnees[
        "CORRECT"
    ] = (

        donnees[
            "OPTION_REPONSE"
        ].notna()

        & option_reponse.eq(
            option_correcte
        )

    ).astype("int8")


    # --------------------------------------------------------
    # 11. Variable explicite de non-réponse
    # --------------------------------------------------------

    donnees[
        "REPONSE_MANQUANTE"
    ] = (
        donnees[
            "OPTION_REPONSE"
        ]
        .isna()
    )


    # --------------------------------------------------------
    # 12. Vérification : 15 lignes par apprenant
    # --------------------------------------------------------

    controle_items = (
        donnees
        .groupby(
            [
                ID,
                "PARCOURS"
            ]
        )
        ["QUESTION_INDEX"]
        .nunique()
    )


    if (
        controle_items > 15
    ).any():

        raise ValueError(
            f"{etape} — plus de 15 items "
            "détectés pour certains apprenants."
        )


    # --------------------------------------------------------
    # 13. Agrégation par apprenant
    # --------------------------------------------------------

    scores = (

        donnees

        .groupby(
            [
                ID,
                "PARCOURS"
            ],
            as_index=False
        )

        .agg(

            N_ITEMS=(
                "QUESTION_INDEX",
                "nunique"
            ),

            N_CORRECTES=(
                "CORRECT",
                "sum"
            ),

            N_REPONSES=(
                "OPTION_REPONSE",
                "count"
            ),

            N_MANQUANTES=(
                "REPONSE_MANQUANTE",
                "sum"
            )

        )

    )


    # --------------------------------------------------------
    # 14. Score sur les 15 items du test
    # --------------------------------------------------------

    scores[
        f"SCORE_{etape}"
    ] = (

        scores[
            "N_CORRECTES"
        ]
        / 15
        * 100

    )


    scores[
        f"COMPLETUDE_{etape}"
    ] = (

        scores[
            "N_REPONSES"
        ]
        / 15
        * 100

    )


    # --------------------------------------------------------
    # 15. Noms explicites
    # --------------------------------------------------------

    scores = scores.rename(
        columns={

            "PARCOURS":
                f"PARCOURS_{etape}",

            "N_ITEMS":
                f"N_ITEMS_{etape}",

            "N_CORRECTES":
                f"N_CORRECTES_{etape}",

            "N_REPONSES":
                f"N_REPONSES_{etape}",

            "N_MANQUANTES":
                f"N_MANQUANTES_{etape}"

        }
    )


    # --------------------------------------------------------
    # 16. Tri
    # --------------------------------------------------------

    scores = (
        scores
        .sort_values(ID)
        .reset_index(drop=True)
    )


    donnees = (
        donnees
        .sort_values(
            [
                ID,
                "PARCOURS",
                "QUESTION_INDEX"
            ]
        )
        .reset_index(drop=True)
    )


    return donnees, scores

# Cellule 11 — Scorer T1 et T2

In [13]:
# ============================================================
# CELLULE 11 — Scoring effectif
# ============================================================

T1_SCORING_LONG, T1_SCORES = (
    scorer_evaluation(
        T1_LONG,
        corrige_t1,
        "T1"
    )
)


T2_SCORING_LONG, T2_SCORES = (
    scorer_evaluation(
        T2_LONG,
        corrige_t2,
        "T2"
    )
)


print(
    "T1 scores :",
    T1_SCORES.shape
)


print(
    "T2 scores :",
    T2_SCORES.shape
)


display(
    T1_SCORES.head(10)
)


display(
    T2_SCORES.head(10)
)

T1 scores : (196, 8)
T2 scores : (206, 8)


,_ID_NORM,PARCOURS_T1,N_ITEMS_T1,N_CORRECTES_T1,N_REPONSES_T1,N_MANQUANTES_T1,SCORE_T1,COMPLETUDE_T1
0,DIEG2026-003,IA,15,14,15,0,93.333333,100.0
1,DIEG2026-007,IA,15,13,15,0,86.666667,100.0
2,DIEG2026-008,BI,15,5,15,0,33.333333,100.0
3,DIEG2026-009,IA,15,11,15,0,73.333333,100.0
4,DIEG2026-010,IA,15,11,15,0,73.333333,100.0
5,DIEG2026-012,IA,15,12,15,0,80.000000,100.0
6,DIEG2026-022,BI,15,10,15,0,66.666667,100.0
7,DIEG2026-025,DS,15,7,15,0,46.666667,100.0
8,DIEG2026-026,BI,15,8,15,0,53.333333,100.0
9,DIEG2026-027,BI,15,13,15,0,86.666667,100.0


,_ID_NORM,PARCOURS_T2,N_ITEMS_T2,N_CORRECTES_T2,N_REPONSES_T2,N_MANQUANTES_T2,SCORE_T2,COMPLETUDE_T2
0,DIEG2026-003,IA,15,14,15,0,93.333333,100.000000
1,DIEG2026-007,IA,15,12,15,0,80.000000,100.000000
2,DIEG2026-008,BI,15,15,15,0,100.000000,100.000000
3,DIEG2026-009,IA,15,14,14,1,93.333333,93.333333
4,DIEG2026-010,IA,15,7,15,0,46.666667,100.000000
5,DIEG2026-012,IA,15,8,14,1,53.333333,93.333333
6,DIEG2026-022,BI,15,6,15,0,40.000000,100.000000
7,DIEG2026-025,DS,15,13,15,0,86.666667,100.000000
8,DIEG2026-026,BI,15,9,13,2,60.000000,86.666667
9,DIEG2026-027,BI,15,9,15,0,60.000000,100.000000


Elle permettra de vérifier qu’on n’a pas corrigé l’erreur Pandas tout en introduisant silencieusement un problème dans les données.

In [14]:
# ============================================================
# CELLULE 11B — Audit technique du scoring T1/T2
# ============================================================

def audit_scoring(
    scoring_long,
    scores,
    etape
):

    print(
        "=" * 75
    )

    print(
        f"AUDIT {etape}"
    )

    print(
        "=" * 75
    )


    print(
        "Lignes détaillées :",
        len(scoring_long)
    )


    print(
        "Apprenants :",
        scores[ID].nunique()
    )


    print(
        "Réponses brutes manquantes :",
        int(
            scoring_long[
                "REPONSE_BRUTE_MANQUANTE"
            ].sum()
        )
    )


    print(
        "Options non reconnues :",
        int(
            scoring_long[
                "OPTION_NON_RECONNUE"
            ].sum()
        )
    )


    print(
        "Réponses correctes :",
        int(
            scoring_long[
                "CORRECT"
            ].sum()
        )
    )


    print(
        "Taux global de réussite :",
        round(
            scoring_long[
                "CORRECT"
            ].mean()
            * 100,
            2
        ),
        "%"
    )


    print(
        "\nDistribution N_ITEMS :"
    )

    display(
        scores[
            f"N_ITEMS_{etape}"
        ]
        .value_counts()
        .sort_index()
        .rename("N_APPRENANTS")
        .to_frame()
    )


    print(
        "\nDistribution N_REPONSES :"
    )

    display(
        scores[
            f"N_REPONSES_{etape}"
        ]
        .value_counts()
        .sort_index()
        .rename("N_APPRENANTS")
        .to_frame()
    )


audit_scoring(
    T1_SCORING_LONG,
    T1_SCORES,
    "T1"
)


audit_scoring(
    T2_SCORING_LONG,
    T2_SCORES,
    "T2"
)

AUDIT T1
Lignes détaillées : 2940
Apprenants : 196
Réponses brutes manquantes : 1
Options non reconnues : 0
Réponses correctes : 2119
Taux global de réussite : 72.07 %

Distribution N_ITEMS :


,N_APPRENANTS
N_ITEMS_T1,
15,196



Distribution N_REPONSES :


,N_APPRENANTS
N_REPONSES_T1,
14,1
15,195


AUDIT T2
Lignes détaillées : 3090
Apprenants : 206
Réponses brutes manquantes : 91
Options non reconnues : 0
Réponses correctes : 2267
Taux global de réussite : 73.37 %

Distribution N_ITEMS :


,N_APPRENANTS
N_ITEMS_T2,
15,206



Distribution N_REPONSES :


,N_APPRENANTS
N_REPONSES_T2,
0,1
2,1
5,1
6,1
11,4
13,5
14,18
15,175


Cellule 11C — KR-20 de T1 et T2

In [15]:
# ============================================================
# CELLULE 11C — Fiabilité interne KR-20 de T1 et T2
# ============================================================

def calculer_kr20_par_parcours(
    scoring_long,
    etape
):

    lignes = []

    colonnes_requises = {
        ID,
        "PARCOURS",
        "QUESTION_INDEX",
        "CORRECT",
        "REPONSE_MANQUANTE"
    }

    manquantes = (
        colonnes_requises
        -
        set(scoring_long.columns)
    )

    assert len(manquantes) == 0, (
        f"{etape} — colonnes manquantes : {manquantes}"
    )


    for parcours, df_p in (
        scoring_long
        .groupby("PARCOURS")
    ):

        # --------------------------------------------
        # Nombre total d'apprenants du parcours
        # --------------------------------------------

        n_total = (
            df_p[ID]
            .nunique()
        )


        # --------------------------------------------
        # Garder uniquement les tests complets
        # pour le calcul de fiabilité
        # --------------------------------------------

        incompletude = (
            df_p
            .groupby(ID)[
                "REPONSE_MANQUANTE"
            ]
            .sum()
        )


        ids_complets = (
            incompletude[
                incompletude == 0
            ]
            .index
        )


        df_complet = (
            df_p[
                df_p[ID]
                .isin(ids_complets)
            ]
            .copy()
        )


        matrice = (
            df_complet
            .pivot(
                index=ID,
                columns="QUESTION_INDEX",
                values="CORRECT"
            )
            .sort_index(
                axis=1
            )
        )


        # --------------------------------------------
        # Contrôles
        # --------------------------------------------

        k = (
            matrice.shape[1]
        )

        n_complets = (
            matrice.shape[0]
        )


        assert k == 15, (
            f"{etape} / {parcours} : "
            f"{k} items au lieu de 15."
        )


        assert (
            matrice
            .notna()
            .all()
            .all()
        ), (
            f"{etape} / {parcours} : "
            "valeurs manquantes inattendues."
        )


        # --------------------------------------------
        # KR-20
        # --------------------------------------------

        p = (
            matrice
            .mean(axis=0)
        )

        q = (
            1 - p
        )


        score_total = (
            matrice
            .sum(axis=1)
        )


        variance_total = (
            score_total
            .var(ddof=1)
        )


        if (
            n_complets < 2
            or
            variance_total <= 0
        ):

            kr20 = np.nan

        else:

            kr20 = (
                k
                /
                (k - 1)
                *
                (
                    1
                    -
                    (
                        (p * q).sum()
                        /
                        variance_total
                    )
                )
            )


        lignes.append(
            {
                "ETAPE":
                    etape,

                "PARCOURS":
                    parcours,

                "N_TOTAL":
                    n_total,

                "N_COMPLETS":
                    n_complets,

                "N_ITEMS":
                    k,

                "VARIANCE_SCORE_TOTAL":
                    variance_total,

                "KR20":
                    kr20
            }
        )


    return (
        pd.DataFrame(
            lignes
        )
        .sort_values(
            [
                "ETAPE",
                "PARCOURS"
            ]
        )
        .reset_index(drop=True)
    )


KR20_T1 = (
    calculer_kr20_par_parcours(
        T1_SCORING_LONG,
        "T1"
    )
)


KR20_T2 = (
    calculer_kr20_par_parcours(
        T2_SCORING_LONG,
        "T2"
    )
)


FIABILITE_T1_T2 = pd.concat(
    [
        KR20_T1,
        KR20_T2
    ],
    ignore_index=True
)


print(
    "Fiabilité interne T1 / T2 :"
)

display(
    FIABILITE_T1_T2.round(3)
)


FIABILITE_T1_T2.to_csv(
    OUTPUT_DIR
    / "fiabilite_KR20_T1_T2.csv",
    index=False,
    encoding="utf-8-sig"
)

Fiabilité interne T1 / T2 :


,ETAPE,PARCOURS,N_TOTAL,N_COMPLETS,N_ITEMS,VARIANCE_SCORE_TOTAL,KR20
0,T1,BI,60,60,15,12.843,0.808
1,T1,DA,32,32,15,9.613,0.809
2,T1,DS,44,44,15,8.632,0.773
3,T1,IA,60,59,15,8.469,0.743
4,T2,BI,60,52,15,9.461,0.758
5,T2,DA,34,28,15,9.698,0.819
6,T2,DS,48,41,15,12.438,0.859
7,T2,IA,64,54,15,16.832,0.887


Cellule 11D — Recouvrement T1/T2

In [16]:
# ============================================================
# CELLULE 11D — Recouvrement des compétences T1 / T2
# ============================================================

import re
import unicodedata


def normaliser_competence(x):

    if pd.isna(x):
        return ""

    x = str(x).strip().lower()

    x = unicodedata.normalize(
        "NFKD",
        x
    )

    x = "".join(
        c
        for c in x
        if not unicodedata.combining(c)
    )

    x = re.sub(
        r"[^a-z0-9]+",
        " ",
        x
    )

    return (
        re.sub(
            r"\s+",
            " ",
            x
        )
        .strip()
    )


T1_COMP = (
    corrige_t1[
        [
            "PARCOURS",
            "QUESTION_INDEX",
            "COMPETENCE"
        ]
    ]
    .copy()
)


T2_COMP = (
    corrige_t2[
        [
            "PARCOURS",
            "QUESTION_INDEX",
            "COMPETENCE"
        ]
    ]
    .copy()
)


for df in [
    T1_COMP,
    T2_COMP
]:

    df["PARCOURS"] = (
        df["PARCOURS"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    df[
        "COMPETENCE_NORM"
    ] = (
        df[
            "COMPETENCE"
        ]
        .map(
            normaliser_competence
        )
    )


lignes = []


for parcours in [
    "DA",
    "BI",
    "DS",
    "IA"
]:

    t1_p = (
        T1_COMP[
            T1_COMP[
                "PARCOURS"
            ] == parcours
        ]
        .copy()
    )


    t2_p = (
        T2_COMP[
            T2_COMP[
                "PARCOURS"
            ] == parcours
        ]
        .copy()
    )


    set_t1 = set(
        t1_p.loc[
            t1_p[
                "COMPETENCE_NORM"
            ] != "",
            "COMPETENCE_NORM"
        ]
    )


    set_t2 = set(
        t2_p.loc[
            t2_p[
                "COMPETENCE_NORM"
            ] != "",
            "COMPETENCE_NORM"
        ]
    )


    intersection = (
        set_t1
        &
        set_t2
    )


    union = (
        set_t1
        |
        set_t2
    )


    # --------------------------------------------
    # Part des items T1 dont la compétence
    # existe aussi dans T2
    # --------------------------------------------

    couverture_items_t1 = (
        t1_p[
            "COMPETENCE_NORM"
        ]
        .isin(set_t2)
        .mean()
    )


    # --------------------------------------------
    # Part des items T2 dont la compétence
    # existait déjà dans T1
    # --------------------------------------------

    couverture_items_t2 = (
        t2_p[
            "COMPETENCE_NORM"
        ]
        .isin(set_t1)
        .mean()
    )


    jaccard = (
        len(intersection)
        /
        len(union)
        if len(union) > 0
        else np.nan
    )


    lignes.append(
        {
            "PARCOURS":
                parcours,

            "N_COMPETENCES_T1":
                len(set_t1),

            "N_COMPETENCES_T2":
                len(set_t2),

            "N_COMPETENCES_COMMUNES":
                len(intersection),

            "JACCARD_COMPETENCES":
                jaccard,

            "PART_ITEMS_T1_RECOUVERTS":
                couverture_items_t1,

            "PART_ITEMS_T2_RECOUVERTS":
                couverture_items_t2,

            "COMPETENCES_COMMUNES":
                " | ".join(
                    sorted(
                        intersection
                    )
                )
        }
    )


RECOUVREMENT_T1_T2 = (
    pd.DataFrame(
        lignes
    )
)


print(
    "Recouvrement T1 / T2 "
    "sur les libellés de compétences :"
)


display(
    RECOUVREMENT_T1_T2[
        [
            "PARCOURS",
            "N_COMPETENCES_T1",
            "N_COMPETENCES_T2",
            "N_COMPETENCES_COMMUNES",
            "JACCARD_COMPETENCES",
            "PART_ITEMS_T1_RECOUVERTS",
            "PART_ITEMS_T2_RECOUVERTS"
        ]
    ].round(3)
)


RECOUVREMENT_T1_T2.to_csv(
    OUTPUT_DIR
    / "recouvrement_competences_T1_T2.csv",
    index=False,
    encoding="utf-8-sig"
)

Recouvrement T1 / T2 sur les libellés de compétences :


,PARCOURS,N_COMPETENCES_T1,N_COMPETENCES_T2,N_COMPETENCES_COMMUNES,JACCARD_COMPETENCES,PART_ITEMS_T1_RECOUVERTS,PART_ITEMS_T2_RECOUVERTS
0,DA,14,15,3,0.115,0.200,0.200
1,BI,15,15,2,0.071,0.133,0.133
2,DS,15,15,3,0.111,0.200,0.200
3,IA,15,15,2,0.071,0.133,0.133


# Cellule 13B
206-175 ,Elle permet de savoir combien des 31 T2 incomplets appartiennent réellement à M_T0 et M_T1.

In [17]:
# ============================================================
# CELLULE 13B — Complétude dans les cohortes prédictives
# ============================================================

t2_completude_cohortes = T2_SCORES.copy()

t2_completude_cohortes["DANS_MT0"] = (
    t2_completude_cohortes[ID]
    .isin(COHORTE_MT0)
)

t2_completude_cohortes["DANS_MT1"] = (
    t2_completude_cohortes[ID]
    .isin(COHORTE_MT1)
)

t2_completude_cohortes["T2_COMPLET"] = (
    t2_completude_cohortes[
        "N_REPONSES_T2"
    ] == 15
)


resume_completude_cohortes = pd.DataFrame({

    "COHORTE": [
        "T2 total",
        "M_T0",
        "M_T1 / comparaison"
    ],

    "N": [
        len(T2_SCORES),
        int(
            t2_completude_cohortes[
                "DANS_MT0"
            ].sum()
        ),
        int(
            t2_completude_cohortes[
                "DANS_MT1"
            ].sum()
        )
    ],

    "N_T2_COMPLETS": [
        int(
            t2_completude_cohortes[
                "T2_COMPLET"
            ].sum()
        ),

        int(
            (
                t2_completude_cohortes[
                    "DANS_MT0"
                ]
                &
                t2_completude_cohortes[
                    "T2_COMPLET"
                ]
            ).sum()
        ),

        int(
            (
                t2_completude_cohortes[
                    "DANS_MT1"
                ]
                &
                t2_completude_cohortes[
                    "T2_COMPLET"
                ]
            ).sum()
        )
    ]
})


resume_completude_cohortes[
    "N_T2_INCOMPLETS"
] = (
    resume_completude_cohortes["N"]
    -
    resume_completude_cohortes["N_T2_COMPLETS"]
)


resume_completude_cohortes[
    "PCT_T2_COMPLETS"
] = (
    resume_completude_cohortes[
        "N_T2_COMPLETS"
    ]
    /
    resume_completude_cohortes[
        "N"
    ]
    * 100
)


display(
    resume_completude_cohortes
)

,COHORTE,N,N_T2_COMPLETS,N_T2_INCOMPLETS,PCT_T2_COMPLETS
0,T2 total,206,175,31,84.951456
1,M_T0,128,108,20,84.375000
2,M_T1 / comparaison,125,105,20,84.000000


# CELLULE 13F — Définition des cohortes expérimentales

In [18]:
# ============================================================
# CELLULE 13F — Définition des cohortes expérimentales
# ============================================================

IDS_T2_COMPLETS = set(
    T2_SCORES.loc[
        T2_SCORES["N_REPONSES_T2"] == 15,
        ID
    ]
)


COHORTE_COMPARAISON_125 = set(
    COHORTE_COMPARAISON
)


COHORTE_COMPARAISON_COMPLETE = (
    COHORTE_COMPARAISON_125
    & IDS_T2_COMPLETS
)


print(
    "Cohorte commune totale :",
    len(COHORTE_COMPARAISON_125)
)

print(
    "Cohorte commune avec T2 complet :",
    len(COHORTE_COMPARAISON_COMPLETE)
)

print(
    "Cas exclus pour T2 incomplet :",
    len(
        COHORTE_COMPARAISON_125
        - COHORTE_COMPARAISON_COMPLETE
    )
)


assert len(COHORTE_COMPARAISON_125) == 125
assert len(COHORTE_COMPARAISON_COMPLETE) == 105

Cohorte commune totale : 125
Cohorte commune avec T2 complet : 105
Cas exclus pour T2 incomplet : 20


# cellule 13G pour regarder la gravité des 20 cas :

In [19]:
# ============================================================
# CELLULE 13G — Sévérité des T2 incomplets dans les 125
# ============================================================

t2_incomplets_125 = (
    T2_SCORES[
        T2_SCORES[ID].isin(
            COHORTE_COMPARAISON_125
        )
        &
        (
            T2_SCORES["N_REPONSES_T2"] < 15
        )
    ]
    .copy()
)


distribution_incompletude_125 = (
    t2_incomplets_125[
        "N_REPONSES_T2"
    ]
    .value_counts()
    .sort_index()
    .rename("N_APPRENANTS")
    .reset_index()
    .rename(
        columns={
            "N_REPONSES_T2":
                "NB_REPONSES"
        }
    )
)


display(
    distribution_incompletude_125
)


display(
    t2_incomplets_125[
        [
            ID,
            "PARCOURS_T2",
            "N_REPONSES_T2",
            "N_MANQUANTES_T2",
            "N_CORRECTES_T2",
            "SCORE_T2",
            "COMPLETUDE_T2"
        ]
    ]
    .sort_values(
        [
            "N_REPONSES_T2",
            ID
        ]
    )
)

,NB_REPONSES,N_APPRENANTS
0,5,1
1,11,2
2,13,2
3,14,15


,_ID_NORM,PARCOURS_T2,N_REPONSES_T2,N_MANQUANTES_T2,N_CORRECTES_T2,SCORE_T2,COMPLETUDE_T2
21,DIEG2026-042,BI,5,10,5,33.333333,33.333333
48,DIEG2026-098,IA,11,4,8,53.333333,73.333333
144,DIEG2026-467,DA,11,4,10,66.666667,73.333333
8,DIEG2026-026,BI,13,2,9,60.000000,86.666667
116,DIEG2026-396,BI,13,2,12,80.000000,86.666667
3,DIEG2026-009,IA,14,1,14,93.333333,93.333333
5,DIEG2026-012,IA,14,1,8,53.333333,93.333333
18,DIEG2026-038,IA,14,1,14,93.333333,93.333333
24,DIEG2026-045,DS,14,1,10,66.666667,93.333333
38,DIEG2026-070,BI,14,1,14,93.333333,93.333333


#  Cellule 13H — Figer les deux analyses

In [20]:
# ============================================================
# CELLULE 13H — Définition officielle des cibles expérimentales
# ============================================================

# Cohorte principale :
# tous les 125 apprenants communs à M_T0 et M_T1.
COHORTE_PRINCIPALE = set(
    COHORTE_COMPARAISON
)


# Cohorte de sensibilité :
# uniquement les apprenants ayant répondu aux 15 questions T2.
COHORTE_SENSIBILITE_COMPLETE = set(
    T2_SCORES.loc[
        T2_SCORES["N_REPONSES_T2"] == 15,
        ID
    ]
) & COHORTE_PRINCIPALE


print(
    "Analyse principale :",
    len(COHORTE_PRINCIPALE)
)

print(
    "Analyse de sensibilité T2 complet :",
    len(COHORTE_SENSIBILITE_COMPLETE)
)


assert len(COHORTE_PRINCIPALE) == 125
assert len(COHORTE_SENSIBILITE_COMPLETE) == 105


# Table de la cible
TARGET_T2 = (
    T2_SCORES[
        T2_SCORES[ID].isin(
            COHORTE_PRINCIPALE
        )
    ]
    .copy()
)


TARGET_T2["TARGET_T2"] = (
    TARGET_T2["SCORE_T2"]
)


TARGET_T2["T2_COMPLET"] = (
    TARGET_T2["N_REPONSES_T2"] == 15
)


TARGET_T2["T2_PRESQUE_COMPLET"] = (
    TARGET_T2["N_REPONSES_T2"] >= 14
)


# Score alternatif :
# uniquement pour analyse de sensibilité descriptive.
TARGET_T2["SCORE_T2_SUR_REPONDU"] = np.where(

    TARGET_T2["N_REPONSES_T2"] > 0,

    (
        TARGET_T2["N_CORRECTES_T2"]
        /
        TARGET_T2["N_REPONSES_T2"]
        * 100
    ),

    np.nan
)


print("\nQualité de la cible :")

display(
    TARGET_T2[
        [
            "T2_COMPLET",
            "T2_PRESQUE_COMPLET"
        ]
    ]
    .value_counts()
    .rename("N")
    .to_frame()
)

Analyse principale : 125
Analyse de sensibilité T2 complet : 105

Qualité de la cible :


N
T2_COMPLET T2_PRESQUE_COMPLET     
True       True                105
False      True                 15
           False                 5

# Maintenant exécutez la Cellule 14
Elle nous donnera la distribution globale de T1 et T2.

In [21]:
# ============================================================
# CELLULE 14 — Statistiques descriptives T1 / T2
# ============================================================

def statistiques_score(df, score):

    return pd.Series({

        "N":
            df[score].count(),

        "Moyenne":
            df[score].mean(),

        "Ecart_type":
            df[score].std(),

        "Minimum":
            df[score].min(),

        "Q1":
            df[score].quantile(0.25),

        "Mediane":
            df[score].median(),

        "Q3":
            df[score].quantile(0.75),

        "Maximum":
            df[score].max()

    })


stats_evaluations = pd.DataFrame({

    "T1":
        statistiques_score(
            T1_SCORES,
            "SCORE_T1"
        ),

    "T2":
        statistiques_score(
            T2_SCORES,
            "SCORE_T2"
        )

})


display(
    stats_evaluations
)


# ------------------------------------------------------------
# Par parcours
# ------------------------------------------------------------

stats_parcours_t1 = (

    T1_SCORES

    .groupby(
        "PARCOURS_T1"
    )

    ["SCORE_T1"]

    .agg(
        N="count",
        MOYENNE="mean",
        ECART_TYPE="std",
        MEDIANE="median",
        MIN="min",
        MAX="max"
    )

)


stats_parcours_t2 = (

    T2_SCORES

    .groupby(
        "PARCOURS_T2"
    )

    ["SCORE_T2"]

    .agg(
        N="count",
        MOYENNE="mean",
        ECART_TYPE="std",
        MEDIANE="median",
        MIN="min",
        MAX="max"
    )

)


print("T1 par parcours")
display(stats_parcours_t1)

print("T2 par parcours")
display(stats_parcours_t2)

,T1,T2
N,196.000000,206.000000
Moyenne,72.074830,73.365696
Ecart_type,21.552376,24.359525
Minimum,13.333333,0.000000
Q1,53.333333,53.333333
Mediane,73.333333,80.000000
Q3,93.333333,93.333333
Maximum,100.000000,100.000000


T1 par parcours


,N,MOYENNE,ECART_TYPE,MEDIANE,MIN,MAX
PARCOURS_T1,,,,,,
BI,60,66.222222,23.891363,66.666667,13.333333,100.0
DA,32,76.666667,20.669788,80.000000,26.666667,100.0
DS,44,78.636364,19.586398,83.333333,26.666667,100.0
IA,60,70.666667,19.492623,73.333333,20.000000,100.0


T2 par parcours


,N,MOYENNE,ECART_TYPE,MEDIANE,MIN,MAX
PARCOURS_T2,,,,,,
BI,60,71.222222,23.240061,73.333333,0.000000,100.0
DA,34,77.843137,22.306140,86.666667,13.333333,100.0
DS,48,76.388889,23.498235,86.666667,20.000000,100.0
IA,64,70.729167,26.893668,80.000000,6.666667,100.0


Cellule 15 — Très importante : parcours T1 ↔ T2
Maintenant nous devons vérifier si un apprenant passe son T2 dans le même parcours que son T1.

In [22]:
# ============================================================
# CELLULE 15 — Cohérence des parcours T1 / T2
# ============================================================

t1_t2 = (

    T1_SCORES

    .merge(

        T2_SCORES,

        on=ID,

        how="inner",

        validate="one_to_one"

    )

)


print(
    "Apprenants T1 + T2 :",
    len(t1_t2)
)


t1_t2["MEME_PARCOURS"] = (

    t1_t2["PARCOURS_T1"]
    ==
    t1_t2["PARCOURS_T2"]

)


print(
    "Même parcours :",
    int(
        t1_t2[
            "MEME_PARCOURS"
        ].sum()
    )
)


print(
    "Parcours différent :",
    int(
        (
            ~t1_t2[
                "MEME_PARCOURS"
            ]
        ).sum()
    )
)


print(
    "Pourcentage même parcours :",
    round(
        t1_t2[
            "MEME_PARCOURS"
        ].mean()
        * 100,
        2
    ),
    "%"
)


print("\nMatrice T1 → T2")

table_parcours = pd.crosstab(

    t1_t2["PARCOURS_T1"],

    t1_t2["PARCOURS_T2"],

    margins=True

)


display(
    table_parcours
)

Apprenants T1 + T2 : 196
Même parcours : 196
Parcours différent : 0
Pourcentage même parcours : 100.0 %

Matrice T1 → T2


PARCOURS_T2,BI,DA,DS,IA,All
PARCOURS_T1,,,,,
BI,60,0,0,0,60
DA,0,32,0,0,32
DS,0,0,44,0,44
IA,0,0,0,60,60
All,60,32,44,60,196


#  Cellule 16 — Cohorte commune finale

In [23]:
# ============================================================
# CELLULE 16 — Table T1/T2 des 125 apprenants
# ============================================================

scores_communs = (

    t1_t2[
        t1_t2[ID]
        .isin(
            COHORTE_PRINCIPALE
        )
    ]

    .copy()

    .sort_values(ID)

    .reset_index(drop=True)

)


assert len(scores_communs) == 125


print(
    "Cohorte principale :",
    len(scores_communs)
)


print(
    "T2 complets :",
    (
        scores_communs[
            "N_REPONSES_T2"
        ] == 15
    ).sum()
)


print(
    "T2 incomplets :",
    (
        scores_communs[
            "N_REPONSES_T2"
        ] < 15
    ).sum()
)


display(
    scores_communs.head()
)

Cohorte principale : 125
T2 complets : 105
T2 incomplets : 20


,_ID_NORM,PARCOURS_T1,N_ITEMS_T1,N_CORRECTES_T1,N_REPONSES_T1,N_MANQUANTES_T1,SCORE_T1,COMPLETUDE_T1,PARCOURS_T2,N_ITEMS_T2,N_CORRECTES_T2,N_REPONSES_T2,N_MANQUANTES_T2,SCORE_T2,COMPLETUDE_T2,MEME_PARCOURS
0,DIEG2026-003,IA,15,14,15,0,93.333333,100.0,IA,15,14,15,0,93.333333,100.000000,True
1,DIEG2026-007,IA,15,13,15,0,86.666667,100.0,IA,15,12,15,0,80.000000,100.000000,True
2,DIEG2026-008,BI,15,5,15,0,33.333333,100.0,BI,15,15,15,0,100.000000,100.000000,True
3,DIEG2026-009,IA,15,11,15,0,73.333333,100.0,IA,15,14,14,1,93.333333,93.333333,True
4,DIEG2026-012,IA,15,12,15,0,80.000000,100.0,IA,15,8,14,1,53.333333,93.333333,True


#  Cellule 17 — Examiner la cible sur les 125 et les 105
Il est préférable de comparer maintenant les distributions des deux cohortes.

In [24]:
# ============================================================
# CELLULE 17 — Distribution de la cible T2
# ============================================================

cible_125 = (
    TARGET_T2["TARGET_T2"]
)


cible_105 = (
    TARGET_T2.loc[
        TARGET_T2[
            "T2_COMPLET"
        ],
        "TARGET_T2"
    ]
)


comparaison_cible = pd.DataFrame({

    "N": [
        cible_125.count(),
        cible_105.count()
    ],

    "MOYENNE": [
        cible_125.mean(),
        cible_105.mean()
    ],

    "ECART_TYPE": [
        cible_125.std(),
        cible_105.std()
    ],

    "MEDIANE": [
        cible_125.median(),
        cible_105.median()
    ],

    "MIN": [
        cible_125.min(),
        cible_105.min()
    ],

    "MAX": [
        cible_125.max(),
        cible_105.max()
    ]

}, index=[

    "Principal_N125",
    "Sensibilite_N105"

])


display(
    comparaison_cible
)

,N,MOYENNE,ECART_TYPE,MEDIANE,MIN,MAX
Principal_N125,125,72.426667,23.727002,80.0,6.666667,100.0
Sensibilite_N105,105,73.650794,23.865380,80.0,6.666667,100.0


In [25]:
# ============================================================
# CELLULE 17B — T2 complet vs incomplet
# ============================================================

comparaison_completude = (

    TARGET_T2

    .groupby(
        "T2_COMPLET"
    )

    ["TARGET_T2"]

    .agg(
        N="count",
        MOYENNE="mean",
        ECART_TYPE="std",
        MEDIANE="median",
        MIN="min",
        MAX="max"
    )

)


display(
    comparaison_completude
)

,N,MOYENNE,ECART_TYPE,MEDIANE,MIN,MAX
T2_COMPLET,,,,,,
False,20,66.000000,22.467651,63.333333,26.666667,93.333333
True,105,73.650794,23.865380,80.000000,6.666667,100.000000


# Cellule 18 — Construire la table maîtresse de modélisation

In [26]:
# ============================================================
# CELLULE 18 — Table maîtresse de modélisation
# ============================================================

# ------------------------------------------------------------
# D1 : préfixage des variables de profil
# ------------------------------------------------------------

d1_model = D1.copy()

d1_model = d1_model[
    d1_model[ID].isin(
        COHORTE_PRINCIPALE
    )
].copy()


d1_model = d1_model.rename(
    columns={
        col: f"P__{col}"
        for col in d1_model.columns
        if col != ID
    }
)


# ------------------------------------------------------------
# T0 : quatre scores diagnostiques
# ------------------------------------------------------------

t0_model = (
    T0_SCORES[
        [
            ID,
            "S_DA",
            "S_BI",
            "S_DS",
            "S_IA"
        ]
    ]
    .copy()
    .rename(
        columns={
            "S_DA": "T0__S_DA",
            "S_BI": "T0__S_BI",
            "S_DS": "T0__S_DS",
            "S_IA": "T0__S_IA"
        }
    )
)


# ------------------------------------------------------------
# T1
# ------------------------------------------------------------

t1_model = (
    T1_SCORES[
        [
            ID,
            "PARCOURS_T1",
            "SCORE_T1",
            "N_REPONSES_T1",
            "COMPLETUDE_T1"
        ]
    ]
    .copy()
    .rename(
        columns={
            "PARCOURS_T1":
                "T1__PARCOURS",

            "SCORE_T1":
                "T1__SCORE",

            "N_REPONSES_T1":
                "T1__N_REPONSES",

            "COMPLETUDE_T1":
                "T1__COMPLETUDE"
        }
    )
)


# ------------------------------------------------------------
# T2 : TARGET uniquement
# ------------------------------------------------------------

target_model = (
    TARGET_T2[
        [
            ID,
            "TARGET_T2",
            "T2_COMPLET",
            "N_REPONSES_T2"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Fusion
# ------------------------------------------------------------

model_base_125 = (
    d1_model
    .merge(
        t0_model,
        on=ID,
        how="inner",
        validate="one_to_one"
    )
    .merge(
        t1_model,
        on=ID,
        how="inner",
        validate="one_to_one"
    )
    .merge(
        target_model,
        on=ID,
        how="inner",
        validate="one_to_one"
    )
)


model_base_125 = (
    model_base_125
    .sort_values(ID)
    .reset_index(drop=True)
)


print(
    "Table principale :",
    model_base_125.shape
)

print(
    "Apprenants uniques :",
    model_base_125[ID].nunique()
)


assert len(model_base_125) == 125
assert model_base_125[ID].nunique() == 125


display(
    model_base_125.head()
)

Table principale : (125, 33)
Apprenants uniques : 125


,P__CreatedAt,P__UpdatedAt,P__IDENTIFICATION,P__Année de naissance,P__Genre,P__Région d'origine,P__Filière,P__Niveau d'étude,P__Statut actuel,P__Niveau informatique,P__Niveau Excel,P__Niveau Power BI,P__Niveau Python,P__Niveau IA,P__Motivation,P__Objectif pro,P__Disponibilité,P__Source info,_ID_NORM,P___ID_NUM,P__ANNEE_NAISSANCE_NUM,P__ANNEE_NAISSANCE_INVALIDE,T0__S_DA,T0__S_BI,T0__S_DS,T0__S_IA,T1__PARCOURS,T1__SCORE,T1__N_REPONSES,T1__COMPLETUDE,TARGET_T2,T2_COMPLET,N_REPONSES_T2
0,2026-04-01 03:01:14+00:00,NaN,DIEG2026-003,2001,Feminin,Diana,Droit / Sciences politiques,Master 2 (M2),Diplome(e) en recherche d emploi,Aucun,Aucun,Aucun,Aucun,Aucun,Ameliorer mon employabilite | Acquerir des com...,Entrepreneur,En semaine - Matin,Facebook,DIEG2026-003,3,2001.0,False,0.0,0.0,0.0,0.0,IA,93.333333,15,100.0,93.333333,True,15
1,2026-04-01 06:30:52+00:00,NaN,DIEG2026-007,2002,Masculin,Sofia,Autre,Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Debutant,Aucun,Intermediaire,Debutant,Acquerir des competences techniques | Realiser...,Developpeur Python / IA,En semaine - Soir | Week-end,WhatsApp,DIEG2026-007,7,2002.0,False,40.0,20.0,80.0,20.0,IA,86.666667,15,100.0,80.000000,True,15
2,2026-04-01 06:34:55+00:00,NaN,DIEG2026-008,2003,Masculin,Diana,"EGS - Economie, Gestion & Sociologie",Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Intermediaire,Debutant,Intermediaire,Debutant,Acquerir des competences techniques | Realiser...,Data Analyst,En semaine - Apres-midi,Facebook,DIEG2026-008,8,2003.0,False,20.0,0.0,20.0,40.0,BI,33.333333,15,100.0,100.000000,True,15
3,2026-04-01 06:35:17+00:00,NaN,DIEG2026-009,2000,Feminin,Fitovinany,"EGS - Economie, Gestion & Sociologie",Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Intermediaire,Debutant,Intermediaire,Debutant,Curiosite | Acquerir des competences technique...,Data Analyst,En semaine - Apres-midi,Facebook,DIEG2026-009,9,2000.0,False,60.0,40.0,20.0,20.0,IA,73.333333,15,100.0,93.333333,False,14
4,2026-04-01 07:40:01+00:00,NaN,DIEG2026-012,2001,Feminin,Diana,IST - Informatique & Sciences des Technologies,Licence 2 (L2),Diplome(e) en recherche d emploi,Intermediaire,Intermediaire,Aucun,Debutant,Intermediaire,Acquerir des competences techniques | Ameliore...,Entrepreneur,Flexible,Ami(e),DIEG2026-012,12,2001.0,False,0.0,0.0,20.0,20.0,IA,80.000000,15,100.0,53.333333,False,14


#  Cellule 19 — Construire aussi la cohorte de sensibilité N=105

In [27]:
# ============================================================
# CELLULE 19 — Cohorte de sensibilité
# ============================================================

model_base_105 = (
    model_base_125[
        model_base_125[
            "T2_COMPLET"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Analyse principale :",
    len(model_base_125)
)


print(
    "Analyse de sensibilité :",
    len(model_base_105)
)


assert len(model_base_125) == 125
assert len(model_base_105) == 105

Analyse principale : 125
Analyse de sensibilité : 105


Cellule 19B — Analyse d'attrition,quand model_base_125 et model_base_105 existent.

In [28]:
# ============================================================
# CELLULE 19B — Analyse de l'attrition / sélection de cohorte
# ============================================================

from scipy.stats import mannwhitneyu


SCORES_T0_ATTRITION = [
    "S_DA",
    "S_BI",
    "S_DS",
    "S_IA"
]


attrition_base = (
    T0_SCORES[
        [
            ID
        ]
        +
        SCORES_T0_ATTRITION
    ]
    .copy()
)


attrition_base[
    "RETENU_N125"
] = (
    attrition_base[ID]
    .isin(
        model_base_125[ID]
    )
)


print(
    "T0 total :",
    len(attrition_base)
)

print(
    "Retenus N=125 :",
    int(
        attrition_base[
            "RETENU_N125"
        ].sum()
    )
)

print(
    "Non retenus :",
    int(
        (
            ~attrition_base[
                "RETENU_N125"
            ]
        ).sum()
    )
)


lignes_attrition = []


for variable in (
    SCORES_T0_ATTRITION
):

    retenus = (
        attrition_base.loc[
            attrition_base[
                "RETENU_N125"
            ],
            variable
        ]
        .dropna()
        .astype(float)
    )


    non_retenus = (
        attrition_base.loc[
            ~attrition_base[
                "RETENU_N125"
            ],
            variable
        ]
        .dropna()
        .astype(float)
    )


    # --------------------------------------------
    # Mann-Whitney
    # --------------------------------------------

    stat, p_value = (
        mannwhitneyu(
            retenus,
            non_retenus,
            alternative="two-sided"
        )
    )


    # --------------------------------------------
    # Différence standardisée des moyennes
    # --------------------------------------------

    var_pool = (
        (
            (len(retenus) - 1)
            *
            retenus.var(ddof=1)
        )
        +
        (
            (len(non_retenus) - 1)
            *
            non_retenus.var(ddof=1)
        )
    ) / (
        len(retenus)
        +
        len(non_retenus)
        -
        2
    )


    sd_pool = np.sqrt(
        var_pool
    )


    smd = (
        (
            retenus.mean()
            -
            non_retenus.mean()
        )
        /
        sd_pool
        if sd_pool > 0
        else np.nan
    )


    lignes_attrition.append(
        {
            "VARIABLE":
                variable,

            "N_RETENUS":
                len(retenus),

            "N_NON_RETENUS":
                len(non_retenus),

            "MOY_RETENUS":
                retenus.mean(),

            "MOY_NON_RETENUS":
                non_retenus.mean(),

            "MEDIANE_RETENUS":
                retenus.median(),

            "MEDIANE_NON_RETENUS":
                non_retenus.median(),

            "DIFF_MOYENNE":
                (
                    retenus.mean()
                    -
                    non_retenus.mean()
                ),

            "SMD":
                smd,

            "U_MANN_WHITNEY":
                stat,

            "P_VALUE":
                p_value
        }
    )


ATTRITION_T0_N125 = (
    pd.DataFrame(
        lignes_attrition
    )
)


print(
    "\nComparaison T0 : "
    "retenus N=125 vs non retenus"
)


display(
    ATTRITION_T0_N125.round(4)
)


ATTRITION_T0_N125.to_csv(
    OUTPUT_DIR
    / "analyse_attrition_T0_N125.csv",
    index=False,
    encoding="utf-8-sig"
)

T0 total : 305
Retenus N=125 : 125
Non retenus : 180

Comparaison T0 : retenus N=125 vs non retenus


,VARIABLE,N_RETENUS,N_NON_RETENUS,MOY_RETENUS,MOY_NON_RETENUS,MEDIANE_RETENUS,MEDIANE_NON_RETENUS,DIFF_MOYENNE,SMD,U_MANN_WHITNEY,P_VALUE
0,S_DA,125,180,26.88,28.1111,20.0,20.0,-1.2311,-0.0440,11016.5,0.7493
1,S_BI,125,180,18.24,23.8889,0.0,20.0,-5.6489,-0.2081,9936.5,0.0639
2,S_DS,125,180,21.92,21.2222,20.0,20.0,0.6978,0.0254,11227.5,0.9753
3,S_IA,125,180,19.20,22.0000,0.0,10.0,-2.8000,-0.1001,10802.5,0.5236


#  Cellule 20 — Audit des variables D1
Cette cellule est essentielle avant de décider ce qui entre dans le modèle.

In [29]:
# ============================================================
# CELLULE 20 — Audit des variables de profil D1
# ============================================================

audit_variables = []


for col in model_base_125.columns:

    if not col.startswith("P__"):
        continue


    serie = model_base_125[col]


    valeurs_non_nulles = (
        serie.dropna()
    )


    exemples = (
        valeurs_non_nulles
        .astype(str)
        .drop_duplicates()
        .head(5)
        .tolist()
    )


    audit_variables.append({

        "VARIABLE":
            col,

        "DTYPE":
            str(serie.dtype),

        "N_UNIQUES":
            serie.nunique(
                dropna=True
            ),

        "N_MANQUANTS":
            int(
                serie.isna().sum()
            ),

        "PCT_MANQUANTS":
            round(
                serie.isna().mean()
                * 100,
                2
            ),

        "EXEMPLES":
            " || ".join(
                exemples
            )

    })


audit_variables = pd.DataFrame(
    audit_variables
)


audit_variables = (
    audit_variables
    .sort_values(
        [
            "N_UNIQUES",
            "VARIABLE"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


display(
    audit_variables
)

,VARIABLE,DTYPE,N_UNIQUES,N_MANQUANTS,PCT_MANQUANTS,EXEMPLES
0,P__CreatedAt,object,125,0,0.0,2026-04-01 03:01:14+00:00 || 2026-04-01 06:30:...
1,P__IDENTIFICATION,object,125,0,0.0,DIEG2026-003 || DIEG2026-007 || DIEG2026-008 |...
2,P___ID_NUM,int64,125,0,0.0,3 || 7 || 8 || 9 || 12
3,P__Motivation,object,61,0,0.0,Ameliorer mon employabilite | Acquerir des com...
4,P__Disponibilité,object,28,4,3.2,En semaine - Matin || En semaine - Soir | Week...
5,P__Année de naissance,int64,19,0,0.0,2001 || 2002 || 2003 || 2000 || 200001022006
6,P__ANNEE_NAISSANCE_NUM,float64,15,4,3.2,2001.0 || 2002.0 || 2003.0 || 2000.0 || 2004.0
7,P__Filière,object,11,0,0.0,Droit / Sciences politiques || Autre || EGS - ...
8,P__Région d'origine,object,11,1,0.8,Diana || Sofia || Fitovinany || Boeny || Sava
9,P__Objectif pro,object,7,1,0.8,Entrepreneur || Developpeur Python / IA || Dat...


# CELLULE 20B — Construction d'une variable AGE propre

In [30]:
# ============================================================
# CELLULE 20B — Construction d'une variable AGE propre
# ============================================================

model_base_125["P__AGE_2026"] = (
    2026
    - pd.to_numeric(
        model_base_125[
            "P__ANNEE_NAISSANCE_NUM"
        ],
        errors="coerce"
    )
)


model_base_105["P__AGE_2026"] = (
    2026
    - pd.to_numeric(
        model_base_105[
            "P__ANNEE_NAISSANCE_NUM"
        ],
        errors="coerce"
    )
)


print("Âge — cohorte 125")

display(
    model_base_125[
        "P__AGE_2026"
    ]
    .describe()
)


print(
    "Valeurs manquantes :",
    model_base_125[
        "P__AGE_2026"
    ]
    .isna()
    .sum()
)

Âge — cohorte 125


count    121.000000
mean      23.644628
std        3.539914
min       18.000000
25%       22.000000
50%       23.000000
75%       25.000000
max       49.000000
Name: P__AGE_2026, dtype: float64

Valeurs manquantes : 4


# Cellule 20C — Examiner Motivation et Disponibilité

In [31]:
# ============================================================
# CELLULE 20C — Modalités élémentaires des champs multi-select
# ============================================================

def extraire_modalites_multiselect(
    serie,
    separateur="|"
):

    modalites = set()

    for valeur in serie.dropna():

        for element in str(valeur).split(
            separateur
        ):

            element = element.strip()

            if element:
                modalites.add(element)

    return sorted(
        modalites
    )


modalites_motivation = (
    extraire_modalites_multiselect(
        model_base_125[
            "P__Motivation"
        ]
    )
)


modalites_disponibilite = (
    extraire_modalites_multiselect(
        model_base_125[
            "P__Disponibilité"
        ]
    )
)


print(
    "Modalités élémentaires Motivation :",
    len(modalites_motivation)
)

for x in modalites_motivation:
    print(" -", x)


print(
    "\nModalités élémentaires Disponibilité :",
    len(modalites_disponibilite)
)

for x in modalites_disponibilite:
    print(" -", x)

Modalités élémentaires Motivation : 7
 - Acquerir des competences techniques
 - Ameliorer mon employabilite
 - Certification
 - Curiosite
 - Evoluer dans mon poste
 - Realiser un projet perso
 - Reconversion

Modalités élémentaires Disponibilité : 6
 - En semaine - Apres-midi
 - En semaine - Matin
 - En semaine - Soir
 - Flexible
 - Samedi
 - Week-end


In [32]:
# ============================================================
# CELLULE 20D — Encodage déterministe des champs multi-select
# ============================================================

MOTIVATIONS = [
    "Acquerir des competences techniques",
    "Ameliorer mon employabilite",
    "Certification",
    "Curiosite",
    "Evoluer dans mon poste",
    "Realiser un projet perso",
    "Reconversion"
]


DISPONIBILITES = [
    "En semaine - Apres-midi",
    "En semaine - Matin",
    "En semaine - Soir",
    "Flexible",
    "Samedi",
    "Week-end"
]


def contient_modalite(valeur, modalite):

    if pd.isna(valeur):
        return np.nan

    elements = {
        x.strip()
        for x in str(valeur).split("|")
        if x.strip()
    }

    return int(
        modalite in elements
    )


def ajouter_multiselects(df):

    df = df.copy()


    # Motivation
    for modalite in MOTIVATIONS:

        nom_col = (
            "P__MOTIVATION__"
            + re.sub(
                r"[^A-Za-z0-9]+",
                "_",
                modalite
            ).strip("_")
        )

        df[nom_col] = (
            df["P__Motivation"]
            .map(
                lambda x:
                    contient_modalite(
                        x,
                        modalite
                    )
            )
        )


    # Disponibilité
    for modalite in DISPONIBILITES:

        nom_col = (
            "P__DISPONIBILITE__"
            + re.sub(
                r"[^A-Za-z0-9]+",
                "_",
                modalite
            ).strip("_")
        )

        df[nom_col] = (
            df["P__Disponibilité"]
            .map(
                lambda x:
                    contient_modalite(
                        x,
                        modalite
                    )
            )
        )


    return df


model_base_125 = ajouter_multiselects(
    model_base_125
)

model_base_105 = ajouter_multiselects(
    model_base_105
)


colonnes_motivation = [
    c
    for c in model_base_125.columns
    if c.startswith(
        "P__MOTIVATION__"
    )
]


colonnes_disponibilite = [
    c
    for c in model_base_125.columns
    if c.startswith(
        "P__DISPONIBILITE__"
    )
]


print(
    "Variables Motivation créées :",
    len(colonnes_motivation)
)

print(
    "Variables Disponibilité créées :",
    len(colonnes_disponibilite)
)


display(
    model_base_125[
        colonnes_motivation
        + colonnes_disponibilite
    ].head()
)

Variables Motivation créées : 7
Variables Disponibilité créées : 6


,P__MOTIVATION__Acquerir_des_competences_techniques,P__MOTIVATION__Ameliorer_mon_employabilite,P__MOTIVATION__Certification,P__MOTIVATION__Curiosite,P__MOTIVATION__Evoluer_dans_mon_poste,P__MOTIVATION__Realiser_un_projet_perso,P__MOTIVATION__Reconversion,P__DISPONIBILITE__En_semaine_Apres_midi,P__DISPONIBILITE__En_semaine_Matin,P__DISPONIBILITE__En_semaine_Soir,P__DISPONIBILITE__Flexible,P__DISPONIBILITE__Samedi,P__DISPONIBILITE__Week_end
0,1,1,0,0,0,1,0,0.0,1.0,0.0,0.0,0.0,0.0
1,1,0,1,1,0,1,0,0.0,0.0,1.0,0.0,0.0,1.0
2,1,0,1,0,0,1,1,1.0,0.0,0.0,0.0,0.0,0.0
3,1,0,1,1,0,1,1,1.0,0.0,0.0,0.0,0.0,0.0
4,1,1,1,0,1,0,1,0.0,0.0,0.0,1.0,0.0,0.0


In [33]:
# ============================================================
# CELLULE 20E — Fréquences des variables multi-select
# ============================================================

resume_multiselect = []


for col in (
    colonnes_motivation
    + colonnes_disponibilite
):

    serie = (
        model_base_125[col]
    )


    resume_multiselect.append({

        "VARIABLE":
            col,

        "N_OUI":
            int(
                (serie == 1).sum()
            ),

        "PCT_OUI":
            round(
                (serie == 1).mean()
                * 100,
                2
            ),

        "N_NON":
            int(
                (serie == 0).sum()
            ),

        "N_MANQUANTS":
            int(
                serie.isna().sum()
            )

    })


resume_multiselect = pd.DataFrame(
    resume_multiselect
)


display(
    resume_multiselect
    .sort_values(
        "PCT_OUI",
        ascending=False
    )
)

,VARIABLE,N_OUI,PCT_OUI,N_NON,N_MANQUANTS
0,P__MOTIVATION__Acquerir_des_competences_techni...,100,80.0,25,0
9,P__DISPONIBILITE__En_semaine_Soir,56,44.8,65,4
3,P__MOTIVATION__Curiosite,49,39.2,76,0
12,P__DISPONIBILITE__Week_end,48,38.4,73,4
1,P__MOTIVATION__Ameliorer_mon_employabilite,47,37.6,78,0
5,P__MOTIVATION__Realiser_un_projet_perso,47,37.6,78,0
2,P__MOTIVATION__Certification,41,32.8,84,0
6,P__MOTIVATION__Reconversion,20,16.0,105,0
4,P__MOTIVATION__Evoluer_dans_mon_poste,19,15.2,106,0
11,P__DISPONIBILITE__Samedi,19,15.2,102,4


0 = Aucun
1 = Débutant
2 = Intermédiaire
3 = Avancé

In [34]:
# ============================================================
# CELLULE 20F — Niveaux auto-déclarés ordonnés
# ============================================================

MAPPING_NIVEAU = {
    "Aucun": 0,
    "Debutant": 1,
    "Intermediaire": 2,
    "Avance": 3
}


VARIABLES_NIVEAUX = [
    "P__Niveau informatique",
    "P__Niveau Excel",
    "P__Niveau Power BI",
    "P__Niveau Python",
    "P__Niveau IA"
]


def ajouter_niveaux_ordonnes(df):

    df = df.copy()

    for col in VARIABLES_NIVEAUX:

        nouveau_nom = (
            col
            .replace(
                "P__Niveau ",
                "P__NIVEAU_"
            )
            .replace(
                " ",
                "_"
            )
            .upper()
        )

        df[
            nouveau_nom
        ] = (
            df[col]
            .map(
                MAPPING_NIVEAU
            )
            .astype("float")
        )

    return df


model_base_125 = (
    ajouter_niveaux_ordonnes(
        model_base_125
    )
)


model_base_105 = (
    ajouter_niveaux_ordonnes(
        model_base_105
    )
)


variables_niveaux_numeriques = [
    c
    for c in model_base_125.columns
    if c.startswith(
        "P__NIVEAU_"
    )
]


display(
    model_base_125[
        variables_niveaux_numeriques
    ].head(10)
)

,P__NIVEAU_INFORMATIQUE,P__NIVEAU_EXCEL,P__NIVEAU_POWER_BI,P__NIVEAU_PYTHON,P__NIVEAU_IA
0,0.0,0.0,0.0,0.0,0.0
1,2.0,1.0,0.0,2.0,1.0
2,2.0,2.0,1.0,2.0,1.0
3,2.0,2.0,1.0,2.0,1.0
4,2.0,2.0,0.0,1.0,2.0
5,1.0,1.0,1.0,1.0,1.0
6,2.0,1.0,0.0,0.0,1.0
7,2.0,1.0,0.0,1.0,1.0
8,2.0,2.0,1.0,0.0,2.0
9,2.0,1.0,0.0,0.0,0.0


#  Cellule 21 — Features finales M_T0 et M_T1

In [35]:
# ============================================================
# CELLULE 21 — Définition finale des variables prédictives
# ============================================================

# Variables numériques / ordinales de profil
FEATURES_PROFIL_NUM = [

    "P__AGE_2026",

    "P__NIVEAU_INFORMATIQUE",
    "P__NIVEAU_EXCEL",
    "P__NIVEAU_POWER_BI",
    "P__NIVEAU_PYTHON",
    "P__NIVEAU_IA",

] + colonnes_motivation + colonnes_disponibilite


# Variables catégorielles structurées
FEATURES_PROFIL_CAT = [

    "P__Genre",
    "P__Région d'origine",
    "P__Filière",
    "P__Niveau d'étude",
    "P__Statut actuel",
    "P__Objectif pro"

]


# Scores diagnostiques objectifs T0
FEATURES_T0 = [

    "T0__S_DA",
    "T0__S_BI",
    "T0__S_DS",
    "T0__S_IA"

]


# Information supplémentaire pour M_T1
FEATURES_T1 = [

    "T1__SCORE"

]


# ------------------------------------------------------------
# Modèle M_T0
# Profil + T0
# ------------------------------------------------------------

FEATURES_MT0_NUM = (
    FEATURES_PROFIL_NUM
    + FEATURES_T0
)

FEATURES_MT0_CAT = (
    FEATURES_PROFIL_CAT
)


# ------------------------------------------------------------
# Modèle M_T1
# Profil + T0 + score T1
# ------------------------------------------------------------

FEATURES_MT1_NUM = (
    FEATURES_PROFIL_NUM
    + FEATURES_T0
    + FEATURES_T1
)

FEATURES_MT1_CAT = (
    FEATURES_PROFIL_CAT
)


print(
    "M_T0 — numériques :",
    len(FEATURES_MT0_NUM)
)

print(
    "M_T0 — catégorielles :",
    len(FEATURES_MT0_CAT)
)

print(
    "M_T1 — numériques :",
    len(FEATURES_MT1_NUM)
)

print(
    "M_T1 — catégorielles :",
    len(FEATURES_MT1_CAT)
)

M_T0 — numériques : 23
M_T0 — catégorielles : 6
M_T1 — numériques : 24
M_T1 — catégorielles : 6


# Cellule 22 — Vérification anti-leakage

In [36]:
# ============================================================
# CELLULE 22 — Contrôle anti-data-leakage
# ============================================================

FEATURES_MT0 = (
    FEATURES_MT0_NUM
    + FEATURES_MT0_CAT
)

FEATURES_MT1 = (
    FEATURES_MT1_NUM
    + FEATURES_MT1_CAT
)


INTERDITES = {

    ID,

    "TARGET_T2",
    "T2_COMPLET",
    "N_REPONSES_T2",

    "SCORE_T2",
    "N_CORRECTES_T2",
    "N_MANQUANTES_T2",
    "COMPLETUDE_T2",
    "PARCOURS_T2",

    "P__Title",
    "P__IDENTIFICATION",
    "P___ID_NUM",

    "P__CreatedAt",
    "P__UpdatedAt",

    "P__Source info",

    "P__Année de naissance",
    "P__ANNEE_NAISSANCE_NUM",
    "P__ANNEE_NAISSANCE_INVALIDE",

    "P__Motivation",
    "P__Disponibilité",

    "P__Niveau informatique",
    "P__Niveau Excel",
    "P__Niveau Power BI",
    "P__Niveau Python",
    "P__Niveau IA"
}


problemes_mt0 = (
    set(FEATURES_MT0)
    & INTERDITES
)

problemes_mt1 = (
    set(FEATURES_MT1)
    & INTERDITES
)


print(
    "Variables interdites dans M_T0 :",
    problemes_mt0
)

print(
    "Variables interdites dans M_T1 :",
    problemes_mt1
)


assert len(problemes_mt0) == 0
assert len(problemes_mt1) == 0


assert "T1__SCORE" not in FEATURES_MT0
assert "T1__SCORE" in FEATURES_MT1


print(
    "✅ Aucun leakage détecté"
)

Variables interdites dans M_T0 : set()
Variables interdites dans M_T1 : set()
✅ Aucun leakage détecté


# Cellule 23 — Création de X et y

In [37]:
# ============================================================
# CELLULE 23 — Matrices X / y
# ============================================================

# ------------------------------------------------------------
# Analyse principale N=125
# ------------------------------------------------------------

X0_125 = (
    model_base_125[
        FEATURES_MT0
    ]
    .copy()
)


X1_125 = (
    model_base_125[
        FEATURES_MT1
    ]
    .copy()
)


y_125 = (
    model_base_125[
        "TARGET_T2"
    ]
    .astype(float)
    .copy()
)


# ------------------------------------------------------------
# Analyse de sensibilité N=105
# ------------------------------------------------------------

X0_105 = (
    model_base_105[
        FEATURES_MT0
    ]
    .copy()
)


X1_105 = (
    model_base_105[
        FEATURES_MT1
    ]
    .copy()
)


y_105 = (
    model_base_105[
        "TARGET_T2"
    ]
    .astype(float)
    .copy()
)


print("ANALYSE PRINCIPALE")

print(
    "X M_T0 :",
    X0_125.shape
)

print(
    "X M_T1 :",
    X1_125.shape
)

print(
    "y      :",
    y_125.shape
)


print("\nSENSIBILITÉ")

print(
    "X M_T0 :",
    X0_105.shape
)

print(
    "X M_T1 :",
    X1_105.shape
)

print(
    "y      :",
    y_105.shape
)

ANALYSE PRINCIPALE
X M_T0 : (125, 29)
X M_T1 : (125, 30)
y      : (125,)

SENSIBILITÉ
X M_T0 : (105, 29)
X M_T1 : (105, 30)
y      : (105,)


# Cellule 24 — Imports ML

In [38]:
# ============================================================
# CELLULE 24 — Bibliothèques Machine Learning
# ============================================================

import sklearn

from sklearn.base import clone

from sklearn.compose import (
    ColumnTransformer
)

from sklearn.pipeline import (
    Pipeline
)

from sklearn.impute import (
    SimpleImputer
)

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.model_selection import (
    KFold,
    RepeatedKFold,
    GridSearchCV
)

from sklearn.dummy import (
    DummyRegressor
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.tree import (
    DecisionTreeRegressor
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


print(
    "scikit-learn :",
    sklearn.__version__
)

scikit-learn : 1.3.2


# Cellule 25 — Prétraitement dans les folds

L'imputation et le scaling ne sont donc jamais calculés sur tout le dataset avant la CV.

In [39]:
# ============================================================
# CORRECTION CELLULE 25 — One-Hot sans multicolinéarité parfaite
# ============================================================

def creer_onehot():

    try:

        return OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False
        )

    except TypeError:

        return OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse=False
        )


def creer_preprocesseur(
    variables_num,
    variables_cat
):

    pipeline_num = Pipeline([

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        )

    ])


    pipeline_cat = Pipeline([

        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            creer_onehot()
        )

    ])


    return ColumnTransformer(

        transformers=[

            (
                "num",
                pipeline_num,
                variables_num
            ),

            (
                "cat",
                pipeline_cat,
                variables_cat
            )

        ],

        remainder="drop"

    )


print(
    "✅ Préprocesseur défini"
)

✅ Préprocesseur défini


# Cellule 26 — Modèles et hyperparamètres

In [40]:
# ============================================================
# CELLULE 26 — Modèles candidats
# ============================================================

MODELES = {

    "Dummy": {

        "model":
            DummyRegressor(),

        "grid": {
            "model__strategy": [
                "mean",
                "median"
            ]
        }
    },


    "LinearRegression": {

        "model":
            LinearRegression(),

        "grid": {}
    },


    "Ridge": {

        "model":
            Ridge(),

        "grid": {
            "model__alpha": [
                0.1,
                1.0,
                10.0,
                100.0
            ]
        }
    },


    "Lasso": {

        "model":
            Lasso(
                max_iter=10000,
                random_state=RANDOM_STATE
            ),

        "grid": {
            "model__alpha": [
                0.01,
                0.1,
                1.0,
                5.0
            ]
        }
    },


    "DecisionTree": {

        "model":
            DecisionTreeRegressor(
                random_state=RANDOM_STATE
            ),

        "grid": {

            "model__max_depth": [
                2,
                3,
                5,
                None
            ],

            "model__min_samples_leaf": [
                2,
                5,
                10
            ]
        }
    },


    "RandomForest": {

        "model":
            RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=1
            ),

        "grid": {

            "model__n_estimators": [
                200,
                400
            ],

            "model__max_depth": [
                3,
                5,
                None
            ],

            "model__min_samples_leaf": [
                2,
                5
            ],

            "model__max_features": [
                "sqrt",
                1.0
            ]
        }
    },


    "GradientBoosting": {

        "model":
            GradientBoostingRegressor(
                random_state=RANDOM_STATE
            ),

        "grid": {

            "model__n_estimators": [
                50,
                100,
                200
            ],

            "model__learning_rate": [
                0.03,
                0.05,
                0.1
            ],

            "model__max_depth": [
                1,
                2,
                3
            ]
        }
    }

}


print(
    "Modèles initiaux :",
    list(MODELES.keys())
)


print(
    "Modèles retenus :",
    list(MODELES.keys())
)

Modèles initiaux : ['Dummy', 'LinearRegression', 'Ridge', 'Lasso', 'DecisionTree', 'RandomForest', 'GradientBoosting']
Modèles retenus : ['Dummy', 'LinearRegression', 'Ridge', 'Lasso', 'DecisionTree', 'RandomForest', 'GradientBoosting']


# Cellule 27 — Folds communs M_T0 / M_T1

Pour N=125, on devrait obtenir environ :
(100, 25) × 5
Le même split sera utilisé pour M_T0 et M_T1.

In [41]:
# ============================================================
# CELLULE 27 — Protocoles de validation
# ============================================================

from sklearn.model_selection import (
    KFold,
    RepeatedKFold
)


OUTER_FOLDS = 5
N_REPEATS = 10


# ============================================================
# A. PROTOCOLE CONFIRMATOIRE
# Repeated CV : 5 folds × 10 répétitions
# ============================================================

outer_cv_125_confirm = RepeatedKFold(
    n_splits=OUTER_FOLDS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)


OUTER_SPLITS_125 = list(
    outer_cv_125_confirm.split(
        X0_125,
        y_125
    )
)


outer_cv_105_confirm = RepeatedKFold(
    n_splits=OUTER_FOLDS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)


OUTER_SPLITS_105 = list(
    outer_cv_105_confirm.split(
        X0_105,
        y_105
    )
)


# ============================================================
# B. PROTOCOLE EXPLORATOIRE
# 5 folds simples pour modèles avancés
# ============================================================

outer_cv_125_explo = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


OUTER_SPLITS_125_EXPLO = list(
    outer_cv_125_explo.split(
        X0_125,
        y_125
    )
)


outer_cv_105_explo = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


OUTER_SPLITS_105_EXPLO = list(
    outer_cv_105_explo.split(
        X0_105,
        y_105
    )
)


print(
    "Confirmatoire N125 :",
    len(OUTER_SPLITS_125)
)

print(
    "Confirmatoire N105 :",
    len(OUTER_SPLITS_105)
)

print(
    "Exploratoire N125 :",
    len(OUTER_SPLITS_125_EXPLO)
)

print(
    "Exploratoire N105 :",
    len(OUTER_SPLITS_105_EXPLO)
)


assert len(OUTER_SPLITS_125) == 50
assert len(OUTER_SPLITS_105) == 50

assert len(OUTER_SPLITS_125_EXPLO) == 5
assert len(OUTER_SPLITS_105_EXPLO) == 5

Confirmatoire N125 : 50
Confirmatoire N105 : 50
Exploratoire N125 : 5
Exploratoire N105 : 5


# Cellule 28 — Fonction Nested CV

In [42]:
# ============================================================
# CELLULE 28 — Fonction Nested CV
# ============================================================

def evaluer_nested_cv(
    X,
    y,
    variables_num,
    variables_cat,
    outer_splits,
    framework,
    analyse
):

    resultats_folds = []
    predictions_oof = []
    parametres = []


    for nom_modele, spec in MODELES.items():

        print(
            f"\n{analyse} | {framework} | {nom_modele}"
        )


        for fold, (
            train_idx,
            test_idx
        ) in enumerate(
            outer_splits,
            start=1
        ):

            X_train = (
                X.iloc[
                    train_idx
                ].copy()
            )

            X_test = (
                X.iloc[
                    test_idx
                ].copy()
            )

            y_train = (
                y.iloc[
                    train_idx
                ].copy()
            )

            y_test = (
                y.iloc[
                    test_idx
                ].copy()
            )


            preprocessor = (
                creer_preprocesseur(
                    variables_num,
                    variables_cat
                )
            )


            pipeline = Pipeline([

                (
                    "preprocess",
                    preprocessor
                ),

                (
                    "model",
                    clone(
                        spec["model"]
                    )
                )

            ])


            inner_cv = KFold(
                n_splits=4,
                shuffle=True,
                random_state=(
                    RANDOM_STATE
                    + fold
                )
            )


            recherche = GridSearchCV(

                estimator=pipeline,

                param_grid=(
                    spec["grid"]
                    if spec["grid"]
                    else [{}]
                ),

                scoring=
                    "neg_mean_absolute_error",

                cv=inner_cv,

                n_jobs=-1,

                refit=True,

                error_score="raise"
            )


            recherche.fit(
                X_train,
                y_train
            )


            pred = (
                recherche.predict(
                    X_test
                )
            )


            mae = mean_absolute_error(
                y_test,
                pred
            )


            rmse = np.sqrt(
                mean_squared_error(
                    y_test,
                    pred
                )
            )


            r2 = r2_score(
                y_test,
                pred
            )


            resultats_folds.append({

                "ANALYSE":
                    analyse,

                "FRAMEWORK":
                    framework,

                "MODELE":
                    nom_modele,

                "FOLD":
                    fold,

                "N_TRAIN":
                    len(train_idx),

                "N_TEST":
                    len(test_idx),

                "MAE":
                    mae,

                "RMSE":
                    rmse,

                "R2":
                    r2

            })


            parametres.append({

                "ANALYSE":
                    analyse,

                "FRAMEWORK":
                    framework,

                "MODELE":
                    nom_modele,

                "FOLD":
                    fold,

                "BEST_PARAMS":
                    json.dumps(
                        recherche.best_params_,
                        ensure_ascii=False
                    )

            })


            for position, (
                idx,
                vrai,
                estime
            ) in enumerate(
                zip(
                    test_idx,
                    y_test,
                    pred
                )
            ):

                predictions_oof.append({

                    "ANALYSE":
                        analyse,

                    "FRAMEWORK":
                        framework,

                    "MODELE":
                        nom_modele,

                    "FOLD":
                        fold,

                    "ROW_INDEX":
                        int(idx),

                    "Y_TRUE":
                        float(vrai),

                    "Y_PRED":
                        float(estime),

                    "ERREUR":
                        float(
                            estime - vrai
                        ),

                    "ABS_ERROR":
                        float(
                            abs(
                                estime - vrai
                            )
                        )

                })


            print(
                f"  Fold {fold} | "
                f"MAE={mae:.3f} | "
                f"RMSE={rmse:.3f} | "
                f"R²={r2:.3f}"
            )


    return (
        pd.DataFrame(
            resultats_folds
        ),
        pd.DataFrame(
            predictions_oof
        ),
        pd.DataFrame(
            parametres
        )
    )

In [43]:
# ============================================================
# CELLULE 28B — Dimension réelle après prétraitement
# ============================================================

prep0 = creer_preprocesseur(
    FEATURES_MT0_NUM,
    FEATURES_MT0_CAT
)

prep1 = creer_preprocesseur(
    FEATURES_MT1_NUM,
    FEATURES_MT1_CAT
)


X0_trans = prep0.fit_transform(
    X0_125
)

X1_trans = prep1.fit_transform(
    X1_125
)


print(
    "M_T0 :",
    X0_125.shape,
    "→ après encodage :",
    X0_trans.shape
)

print(
    "M_T1 :",
    X1_125.shape,
    "→ après encodage :",
    X1_trans.shape
)

M_T0 : (125, 29) → après encodage : (125, 60)
M_T1 : (125, 30) → après encodage : (125, 61)


In [44]:
# ============================================================
# Cellule 28C— Cache des calculs Nested CV coûteux
# ============================================================

CACHE_DIR = (
    OUTPUT_DIR
    / "cache_nested_cv"
)


CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# False :
# recharge les résultats déjà calculés.
#
# True :
# force un nouvel entraînement complet.
RECALCULER_NESTED_CV = False


def chemins_cache(prefixe):

    return {

        "resultats":
            CACHE_DIR
            / f"{prefixe}_resultats.csv",

        "oof":
            CACHE_DIR
            / f"{prefixe}_oof.csv",

        "params":
            CACHE_DIR
            / f"{prefixe}_params.csv"
    }


def cache_disponible(
    prefixe
):

    fichiers = (
        chemins_cache(
            prefixe
        )
    )

    return all(
        fichier.exists()
        for fichier
        in fichiers.values()
    )


def sauvegarder_cache(
    prefixe,
    resultats,
    oof,
    params
):

    fichiers = (
        chemins_cache(
            prefixe
        )
    )


    resultats.to_csv(
        fichiers[
            "resultats"
        ],
        index=False,
        encoding="utf-8-sig"
    )


    oof.to_csv(
        fichiers[
            "oof"
        ],
        index=False,
        encoding="utf-8-sig"
    )


    params.to_csv(
        fichiers[
            "params"
        ],
        index=False,
        encoding="utf-8-sig"
    )


def charger_cache(
    prefixe
):

    fichiers = (
        chemins_cache(
            prefixe
        )
    )


    return (

        pd.read_csv(
            fichiers[
                "resultats"
            ]
        ),

        pd.read_csv(
            fichiers[
                "oof"
            ]
        ),

        pd.read_csv(
            fichiers[
                "params"
            ]
        )
    )


print(
    "RECALCULER_NESTED_CV =",
    RECALCULER_NESTED_CV
)

print(
    "Cache :",
    CACHE_DIR
)

RECALCULER_NESTED_CV = False
Cache : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/03_modelisation_MT0_MT1/cache_nested_cv


In [45]:
# ============================================================
# CELLULE 28D — Vérification du cache disponible
# ============================================================

for prefixe in [
    "MT0_N125",
    "MT1_N125",
    "MT0_N105",
    "MT1_N105"
]:

    print(
        prefixe,
        "->",
        "CACHE DISPONIBLE"
        if cache_disponible(prefixe)
        else "CACHE ABSENT"
    )

MT0_N125 -> CACHE DISPONIBLE
MT1_N125 -> CACHE DISPONIBLE
MT0_N105 -> CACHE DISPONIBLE
MT1_N105 -> CACHE DISPONIBLE


In [46]:
# ============================================================
# SUPPRESSION CIBLEE DU CACHE M_T1 N=105 ERRONE
# A exécuter une seule fois
# ============================================================

fichiers_cache_mt1_105 = (
    chemins_cache(
        "MT1_N105"
    )
)


for nom, fichier in (
    fichiers_cache_mt1_105.items()
):

    if fichier.exists():

        fichier.unlink()

        print(
            "Supprimé :",
            fichier
        )

    else:

        print(
            "Déjà absent :",
            fichier
        )


print(
    "\nCache MT1_N105 réinitialisé."
)

Supprimé : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/03_modelisation_MT0_MT1/cache_nested_cv/MT1_N105_resultats.csv
Supprimé : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/03_modelisation_MT0_MT1/cache_nested_cv/MT1_N105_oof.csv
Supprimé : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/03_modelisation_MT0_MT1/cache_nested_cv/MT1_N105_params.csv

Cache MT1_N105 réinitialisé.


# Cellule 29 — Lancer l'analyse principale N=125

In [47]:
# ============================================================
# CELLULE 29 — Nested CV principale N=125 avec cache
# ============================================================

# ------------------------------------------------------------
# M_T0
# ------------------------------------------------------------

if (
    not RECALCULER_NESTED_CV
    and
    cache_disponible(
        "MT0_N125"
    )
):

    print(
        "Chargement cache M_T0 N=125"
    )

    (
        RESULTATS_MT0_125,
        OOF_MT0_125,
        PARAMS_MT0_125
    ) = charger_cache(
        "MT0_N125"
    )


else:

    print(
        "Calcul M_T0 N=125..."
    )

    (
        RESULTATS_MT0_125,
        OOF_MT0_125,
        PARAMS_MT0_125
    ) = evaluer_nested_cv(

        X=X0_125,

        y=y_125,

        variables_num=
            FEATURES_MT0_NUM,

        variables_cat=
            FEATURES_MT0_CAT,

        outer_splits=
            OUTER_SPLITS_125,

        framework=
            "M_T0",

        analyse=
            "PRINCIPALE_N125"
    )


    sauvegarder_cache(

        "MT0_N125",

        RESULTATS_MT0_125,

        OOF_MT0_125,

        PARAMS_MT0_125
    )


# ------------------------------------------------------------
# M_T1
# ------------------------------------------------------------

if (
    not RECALCULER_NESTED_CV
    and
    cache_disponible(
        "MT1_N125"
    )
):

    print(
        "Chargement cache M_T1 N=125"
    )

    (
        RESULTATS_MT1_125,
        OOF_MT1_125,
        PARAMS_MT1_125
    ) = charger_cache(
        "MT1_N125"
    )


else:

    print(
        "Calcul M_T1 N=125..."
    )

    (
        RESULTATS_MT1_125,
        OOF_MT1_125,
        PARAMS_MT1_125
    ) = evaluer_nested_cv(

        X=X1_125,

        y=y_125,

        variables_num=
            FEATURES_MT1_NUM,

        variables_cat=
            FEATURES_MT1_CAT,

        outer_splits=
            OUTER_SPLITS_125,

        framework=
            "M_T1",

        analyse=
            "PRINCIPALE_N125"
    )


    sauvegarder_cache(

        "MT1_N125",

        RESULTATS_MT1_125,

        OOF_MT1_125,

        PARAMS_MT1_125
    )


print(
    "\nAnalyse principale N=125 terminée."
)

Chargement cache M_T0 N=125


Chargement cache M_T1 N=125

Analyse principale N=125 terminée.


# Cellule 31 — Comparaison appariée M_T0 vs M_T1

In [48]:
# ============================================================
# CELLULE 31 — Différence M_T1 - M_T0 sur les mêmes folds
# ============================================================

comparaison_125 = (

    RESULTATS_MT0_125[
        [
            "MODELE",
            "FOLD",
            "MAE",
            "RMSE",
            "R2"
        ]
    ]

    .merge(

        RESULTATS_MT1_125[
            [
                "MODELE",
                "FOLD",
                "MAE",
                "RMSE",
                "R2"
            ]
        ],

        on=[
            "MODELE",
            "FOLD"
        ],

        suffixes=(
            "_MT0",
            "_MT1"
        ),

        validate="one_to_one"

    )

)


comparaison_125[
    "DELTA_MAE"
] = (
    comparaison_125[
        "MAE_MT1"
    ]
    -
    comparaison_125[
        "MAE_MT0"
    ]
)


comparaison_125[
    "DELTA_RMSE"
] = (
    comparaison_125[
        "RMSE_MT1"
    ]
    -
    comparaison_125[
        "RMSE_MT0"
    ]
)


comparaison_125[
    "DELTA_R2"
] = (
    comparaison_125[
        "R2_MT1"
    ]
    -
    comparaison_125[
        "R2_MT0"
    ]
)


resume_comparaison_125 = (

    comparaison_125

    .groupby(
        "MODELE",
        as_index=False
    )

    .agg(

        DELTA_MAE_MOYEN=(
            "DELTA_MAE",
            "mean"
        ),

        DELTA_RMSE_MOYEN=(
            "DELTA_RMSE",
            "mean"
        ),

        DELTA_R2_MOYEN=(
            "DELTA_R2",
            "mean"
        )

    )

)


display(
    resume_comparaison_125
)

,MODELE,DELTA_MAE_MOYEN,DELTA_RMSE_MOYEN,DELTA_R2_MOYEN
0,DecisionTree,-3.399318e+00,-2.732280e+00,2.275009e-01
1,Dummy,0.000000e+00,0.000000e+00,0.000000e+00
2,GradientBoosting,-2.547221e+00,-2.166076e+00,1.847105e-01
3,Lasso,-2.817418e+00,-2.445280e+00,2.082225e-01
4,LinearRegression,-1.889277e+12,-7.068263e+12,2.767682e+24
5,RandomForest,-3.239443e+00,-2.621807e+00,2.086262e-01
6,Ridge,-1.499866e+00,-1.300026e+00,1.056749e-01


# Cellule 32 — Nested CV N=105

Passons maintenant à l’analyse de sensibilité sur les 105 T2 complets.

In [49]:
# ============================================================
# CELLULE 32 — Sensibilité N=105 avec cache
# ============================================================


# ============================================================
# 1. M_T0 — Profil + T0
# ============================================================

if (
    not RECALCULER_NESTED_CV
    and
    cache_disponible(
        "MT0_N105"
    )
):

    print(
        "Chargement cache M_T0 N=105"
    )

    (
        RESULTATS_MT0_105,
        OOF_MT0_105,
        PARAMS_MT0_105
    ) = charger_cache(
        "MT0_N105"
    )


else:

    print(
        "Calcul M_T0 N=105..."
    )

    (
        RESULTATS_MT0_105,
        OOF_MT0_105,
        PARAMS_MT0_105
    ) = evaluer_nested_cv(

        X=X0_105,

        y=y_105,

        variables_num=
            FEATURES_MT0_NUM,

        variables_cat=
            FEATURES_MT0_CAT,

        outer_splits=
            OUTER_SPLITS_105,

        framework=
            "M_T0",

        analyse=
            "SENSIBILITE_N105"
    )


    sauvegarder_cache(

        "MT0_N105",

        RESULTATS_MT0_105,

        OOF_MT0_105,

        PARAMS_MT0_105
    )



# ============================================================
# 2. M_T1 — Profil + T0 + T1
# ============================================================

if (
    not RECALCULER_NESTED_CV
    and
    cache_disponible(
        "MT1_N105"
    )
):

    print(
        "Chargement cache M_T1 N=105"
    )

    (
        RESULTATS_MT1_105,
        OOF_MT1_105,
        PARAMS_MT1_105
    ) = charger_cache(
        "MT1_N105"
    )


else:

    print(
        "Calcul M_T1 N=105..."
    )

    (
        RESULTATS_MT1_105,
        OOF_MT1_105,
        PARAMS_MT1_105
    ) = evaluer_nested_cv(

        X=X1_105,

        y=y_105,

        variables_num=
            FEATURES_MT1_NUM,

        variables_cat=
            FEATURES_MT1_CAT,

        outer_splits=
            OUTER_SPLITS_105,

        framework=
            "M_T1",

        analyse=
            "SENSIBILITE_N105"
    )


    sauvegarder_cache(

        "MT1_N105",

        RESULTATS_MT1_105,

        OOF_MT1_105,

        PARAMS_MT1_105
    )



# ============================================================
# 3. Vérifications
# ============================================================

assert (
    "RESULTATS_MT0_105"
    in globals()
)

assert (
    "RESULTATS_MT1_105"
    in globals()
)


assert len(
    RESULTATS_MT0_105
) > 0


assert len(
    RESULTATS_MT1_105
) > 0


print(
    "\nAnalyse de sensibilité N=105 terminée."
)


print(
    "M_T0 :",
    RESULTATS_MT0_105.shape
)

print(
    "M_T1 :",
    RESULTATS_MT1_105.shape
)

Chargement cache M_T0 N=105


Calcul M_T1 N=105...

SENSIBILITE_N105 | M_T1 | Dummy
  Fold 1 | MAE=17.460 | RMSE=20.728 | R²=-0.029
  Fold 2 | MAE=18.095 | RMSE=22.015 | R²=-0.064
  Fold 3 | MAE=20.000 | RMSE=26.146 | R²=-0.026
  Fold 4 | MAE=21.905 | RMSE=27.255 | R²=-0.173
  Fold 5 | MAE=20.952 | RMSE=26.105 | R²=-0.111
  Fold 6 | MAE=19.683 | RMSE=23.002 | R²=-0.176
  Fold 7 | MAE=19.683 | RMSE=26.186 | R²=-0.009
  Fold 8 | MAE=22.857 | RMSE=28.877 | R²=-0.240
  Fold 9 | MAE=16.667 | RMSE=19.368 | R²=-0.192
  Fold 10 | MAE=21.587 | RMSE=25.281 | R²=-0.295
  Fold 11 | MAE=17.778 | RMSE=23.185 | R²=-0.050
  Fold 12 | MAE=22.857 | RMSE=28.284 | R²=-0.170
  Fold 13 | MAE=23.810 | RMSE=27.410 | R²=-0.330
  Fold 14 | MAE=16.508 | RMSE=22.817 | R²=-0.007
  Fold 15 | MAE=17.460 | RMSE=20.315 | R²=-0.000
  Fold 16 | MAE=21.270 | RMSE=24.731 | R²=-0.078
  Fold 17 | MAE=17.460 | RMSE=20.728 | R²=-0.012
  Fold 18 | MAE=23.605 | RMSE=28.933 | R²=-0.038
  Fold 19 | MAE=20.317 | RMSE=25.531 | R²=-0.041
  Fold 20 | MAE=16.508 |

# Cellule 33 — Performances absolues N=105

In [50]:
# ============================================================
# CELLULE 33 — Synthèse N=105
# ============================================================

resultats_105 = pd.concat(
    [
        RESULTATS_MT0_105,
        RESULTATS_MT1_105
    ],
    ignore_index=True
)


resume_105 = (

    resultats_105

    .groupby(
        [
            "FRAMEWORK",
            "MODELE"
        ],
        as_index=False
    )

    .agg(

        MAE_MOYEN=("MAE", "mean"),
        MAE_STD=("MAE", "std"),

        RMSE_MOYEN=("RMSE", "mean"),
        RMSE_STD=("RMSE", "std"),

        R2_MOYEN=("R2", "mean"),
        R2_STD=("R2", "std")

    )
)


display(
    resume_105.sort_values(
        [
            "FRAMEWORK",
            "MAE_MOYEN"
        ]
    )
)

,FRAMEWORK,MODELE,MAE_MOYEN,MAE_STD,RMSE_MOYEN,RMSE_STD,R2_MOYEN,R2_STD
1,M_T0,Dummy,1.973878e+01,3.139220e+00,2.426286e+01,3.894017e+00,-1.075279e-01,1.135031e-01
3,M_T0,Lasso,2.034681e+01,2.923246e+00,2.393439e+01,3.432208e+00,-9.939128e-02,2.481145e-01
6,M_T0,Ridge,2.038374e+01,2.654246e+00,2.418001e+01,3.227049e+00,-1.256778e-01,2.574259e-01
5,M_T0,RandomForest,2.040223e+01,2.529690e+00,2.379471e+01,3.390123e+00,-7.807058e-02,1.854788e-01
2,M_T0,GradientBoosting,2.142603e+01,2.707691e+00,2.539449e+01,3.906072e+00,-2.340192e-01,2.671825e-01
0,M_T0,DecisionTree,2.214683e+01,3.383974e+00,2.660336e+01,4.172194e+00,-3.583451e-01,3.245449e-01
4,M_T0,LinearRegression,1.802847e+11,1.274805e+12,8.042146e+11,5.686656e+12,-9.188842e+22,6.497492e+23
7,M_T1,DecisionTree,1.588085e+01,2.654825e+00,2.045354e+01,2.962868e+00,1.921672e-01,1.892827e-01
12,M_T1,RandomForest,1.621935e+01,2.221532e+00,1.999529e+01,2.707394e+00,2.332501e-01,1.499172e-01
9,M_T1,GradientBoosting,1.764402e+01,2.522628e+00,2.160554e+01,3.238311e+00,1.026738e-01,2.108911e-01


# Cellule 34 — Comparaison M_T1 vs M_T0 sur N=105

In [51]:
# ============================================================
# CELLULE 34 — Comparaison appariée N=105
# ============================================================

comparaison_105 = (

    RESULTATS_MT0_105[
        [
            "MODELE",
            "FOLD",
            "MAE",
            "RMSE",
            "R2"
        ]
    ]

    .merge(

        RESULTATS_MT1_105[
            [
                "MODELE",
                "FOLD",
                "MAE",
                "RMSE",
                "R2"
            ]
        ],

        on=[
            "MODELE",
            "FOLD"
        ],

        suffixes=(
            "_MT0",
            "_MT1"
        ),

        validate="one_to_one"
    )
)


comparaison_105[
    "DELTA_MAE"
] = (
    comparaison_105["MAE_MT1"]
    -
    comparaison_105["MAE_MT0"]
)


comparaison_105[
    "DELTA_RMSE"
] = (
    comparaison_105["RMSE_MT1"]
    -
    comparaison_105["RMSE_MT0"]
)


comparaison_105[
    "DELTA_R2"
] = (
    comparaison_105["R2_MT1"]
    -
    comparaison_105["R2_MT0"]
)


resume_comparaison_105 = (

    comparaison_105

    .groupby(
        "MODELE",
        as_index=False
    )

    .agg(

        DELTA_MAE_MOYEN=(
            "DELTA_MAE",
            "mean"
        ),

        DELTA_RMSE_MOYEN=(
            "DELTA_RMSE",
            "mean"
        ),

        DELTA_R2_MOYEN=(
            "DELTA_R2",
            "mean"
        )

    )
)


display(
    resume_comparaison_105
)

,MODELE,DELTA_MAE_MOYEN,DELTA_RMSE_MOYEN,DELTA_R2_MOYEN
0,DecisionTree,-6.265973e+00,-6.149813e+00,5.505122e-01
1,Dummy,-3.552714e-17,-4.263256e-16,-3.854989e-17
2,GradientBoosting,-3.782011e+00,-3.788943e+00,3.366930e-01
3,Lasso,-2.681441e+00,-2.094388e+00,1.809672e-01
4,LinearRegression,-1.802847e+11,-8.042146e+11,9.188842e+22
5,RandomForest,-4.182878e+00,-3.799425e+00,3.113207e-01
6,Ridge,-1.568322e+00,-1.232242e+00,1.093592e-01


In [52]:
# ============================================================
# CELLULE 35 — Agrégation des prédictions OOF répétées
# ============================================================

def agreger_oof_repete(df):

    resultat = (
        df
        .groupby(
            [
                "ANALYSE",
                "FRAMEWORK",
                "MODELE",
                "ROW_INDEX"
            ],
            as_index=False
        )
        .agg(
            Y_TRUE=(
                "Y_TRUE",
                "first"
            ),

            Y_PRED=(
                "Y_PRED",
                "mean"
            ),

            Y_PRED_SD=(
                "Y_PRED",
                "std"
            ),

            N_PREDICTIONS=(
                "Y_PRED",
                "count"
            )
        )
    )

    resultat[
        "ERREUR"
    ] = (
        resultat["Y_PRED"]
        -
        resultat["Y_TRUE"]
    )

    resultat[
        "ABS_ERROR"
    ] = (
        resultat["ERREUR"]
        .abs()
    )

    return resultat


OOF_MT0_125_AGG = agreger_oof_repete(
    OOF_MT0_125
)

OOF_MT1_125_AGG = agreger_oof_repete(
    OOF_MT1_125
)

OOF_MT0_105_AGG = agreger_oof_repete(
    OOF_MT0_105
)

OOF_MT1_105_AGG = agreger_oof_repete(
    OOF_MT1_105
)


assert (
    OOF_MT1_125_AGG[
        "ROW_INDEX"
    ].nunique()
    == 125
)

assert (
    OOF_MT1_105_AGG[
        "ROW_INDEX"
    ].nunique()
    == 105
)


OOF_105_AGG = pd.concat(
    [
        OOF_MT0_105_AGG,
        OOF_MT1_105_AGG
    ],
    ignore_index=True
)


resume_oof_105 = (
    OOF_105_AGG

    .groupby(
        [
            "FRAMEWORK",
            "MODELE"
        ],
        as_index=False
    )

    .agg(
        N=(
            "Y_TRUE",
            "count"
        ),

        MAE_OOF=(
            "ABS_ERROR",
            "mean"
        ),

        MEDIAN_ABS_ERROR=(
            "ABS_ERROR",
            "median"
        )
    )
)


display(
    resume_oof_105.sort_values(
        [
            "FRAMEWORK",
            "MAE_OOF"
        ]
    )
)

,FRAMEWORK,MODELE,N,MAE_OOF,MEDIAN_ABS_ERROR
1,M_T0,Dummy,105,1.973878e+01,20.000000
6,M_T0,Ridge,105,1.996402e+01,17.896758
3,M_T0,Lasso,105,2.007297e+01,18.467474
5,M_T0,RandomForest,105,2.030323e+01,19.132413
2,M_T0,GradientBoosting,105,2.092414e+01,19.132982
0,M_T0,DecisionTree,105,2.130850e+01,18.293002
4,M_T0,LinearRegression,105,1.802847e+11,25.572726
7,M_T1,DecisionTree,105,1.546067e+01,13.207103
12,M_T1,RandomForest,105,1.590910e+01,13.389799
9,M_T1,GradientBoosting,105,1.702413e+01,14.932642


Cellule 35B — Test propre de H1 : M_T0 vs baseline moyenne

In [53]:
# ============================================================
# CELLULE 35B — Test de H1 : M_T0 vs référence moyenne
# ============================================================

from scipy.stats import wilcoxon


N_BOOTSTRAP_H1 = 5000


# ============================================================
# 1. Construire la baseline moyenne avec EXACTEMENT
#    les mêmes partitions externes
# ============================================================

def oof_dummy_moyenne(
    y,
    outer_splits,
    analyse
):

    lignes = []


    for fold, (
        train_idx,
        test_idx
    ) in enumerate(
        outer_splits,
        start=1
    ):

        y_train = (
            y.iloc[
                train_idx
            ]
            .astype(float)
        )


        moyenne_train = (
            y_train.mean()
        )


        y_test = (
            y.iloc[
                test_idx
            ]
            .astype(float)
        )


        for row_index, y_true in zip(
            test_idx,
            y_test
        ):

            lignes.append(
                {
                    "ANALYSE":
                        analyse,

                    "FRAMEWORK":
                        "M_T0",

                    "MODELE":
                        "DummyMean",

                    "FOLD":
                        fold,

                    "ROW_INDEX":
                        int(
                            row_index
                        ),

                    "Y_TRUE":
                        float(
                            y_true
                        ),

                    "Y_PRED":
                        float(
                            moyenne_train
                        )
                }
            )


    brut = pd.DataFrame(
        lignes
    )


    agrege = (
        brut
        .groupby(
            "ROW_INDEX",
            as_index=False
        )
        .agg(
            Y_TRUE=(
                "Y_TRUE",
                "first"
            ),

            Y_PRED=(
                "Y_PRED",
                "mean"
            ),

            Y_PRED_SD=(
                "Y_PRED",
                "std"
            ),

            N_PREDICTIONS=(
                "Y_PRED",
                "count"
            )
        )
    )


    agrege[
        "ERREUR"
    ] = (
        agrege[
            "Y_PRED"
        ]
        -
        agrege[
            "Y_TRUE"
        ]
    )


    agrege[
        "ABS_ERROR"
    ] = (
        agrege[
            "ERREUR"
        ]
        .abs()
    )


    return brut, agrege


# ============================================================
# 2. Baselines N=125 et N=105
# ============================================================

(
    OOF_DUMMY_MEAN_125,
    OOF_DUMMY_MEAN_125_AGG
) = oof_dummy_moyenne(
    y=y_125,
    outer_splits=OUTER_SPLITS_125,
    analyse="H1_N125"
)


(
    OOF_DUMMY_MEAN_105,
    OOF_DUMMY_MEAN_105_AGG
) = oof_dummy_moyenne(
    y=y_105,
    outer_splits=OUTER_SPLITS_105,
    analyse="H1_N105"
)


# ============================================================
# 3. Comparaison RandomForest M_T0 vs moyenne
# ============================================================

def tester_h1(
    oof_mt0_agrege,
    oof_dummy_agrege,
    cohorte,
    random_state=42
):

    rf = (
        oof_mt0_agrege[
            oof_mt0_agrege[
                "MODELE"
            ] == "RandomForest"
        ][
            [
                "ROW_INDEX",
                "Y_TRUE",
                "ABS_ERROR"
            ]
        ]
        .rename(
            columns={
                "ABS_ERROR":
                    "ABS_ERROR_RF"
            }
        )
        .copy()
    )


    dummy = (
        oof_dummy_agrege[
            [
                "ROW_INDEX",
                "Y_TRUE",
                "ABS_ERROR"
            ]
        ]
        .rename(
            columns={
                "ABS_ERROR":
                    "ABS_ERROR_DUMMY"
            }
        )
        .copy()
    )


    paire = (
        rf
        .merge(
            dummy,
            on="ROW_INDEX",
            how="inner",
            suffixes=(
                "_RF",
                "_DUMMY"
            ),
            validate="one_to_one"
        )
    )


    assert len(
        paire
    ) == cohorte


    # Gain positif =
    # RandomForest meilleur que moyenne

    paire[
        "GAIN_RF_VS_DUMMY"
    ] = (
        paire[
            "ABS_ERROR_DUMMY"
        ]
        -
        paire[
            "ABS_ERROR_RF"
        ]
    )


    gains = (
        paire[
            "GAIN_RF_VS_DUMMY"
        ]
        .to_numpy(
            dtype=float
        )
    )


    # --------------------------------------------
    # Bootstrap IC95
    # --------------------------------------------

    rng = (
        np.random.default_rng(
            random_state
            +
            cohorte
        )
    )


    boot = np.empty(
        N_BOOTSTRAP_H1
    )


    n = len(
        gains
    )


    for b in range(
        N_BOOTSTRAP_H1
    ):

        indices = (
            rng.integers(
                0,
                n,
                size=n
            )
        )


        boot[b] = (
            gains[
                indices
            ]
            .mean()
        )


    ic_bas = (
        np.percentile(
            boot,
            2.5
        )
    )


    ic_haut = (
        np.percentile(
            boot,
            97.5
        )
    )


    # --------------------------------------------
    # Wilcoxon directionnel
    # H1 : erreur Dummy > erreur RF
    # --------------------------------------------

    erreurs_dummy = (
        paire[
            "ABS_ERROR_DUMMY"
        ]
        .to_numpy(
            dtype=float
        )
    )


    erreurs_rf = (
        paire[
            "ABS_ERROR_RF"
        ]
        .to_numpy(
            dtype=float
        )
    )


    if np.allclose(
        erreurs_dummy,
        erreurs_rf
    ):

        stat = 0.0
        p_value = 1.0

    else:

        stat, p_value = wilcoxon(
            erreurs_dummy,
            erreurs_rf,
            alternative="greater"
        )


    resultat = {
        "COHORTE":
            cohorte,

        "N":
            len(
                paire
            ),

        "MAE_DUMMY_MEAN":
            erreurs_dummy.mean(),

        "MAE_RANDOMFOREST_MT0":
            erreurs_rf.mean(),

        "GAIN_MAE_RF_VS_DUMMY":
            gains.mean(),

        "IC95_BAS":
            ic_bas,

        "IC95_HAUT":
            ic_haut,

        "IC95_EXCLUT_0_POSITIVEMENT":
            ic_bas > 0,

        "STAT_WILCOXON":
            stat,

        "P_VALUE_UNILATERAL":
            p_value,

        "H1_SOUTENUE":
            (
                gains.mean() > 0
                and
                ic_bas > 0
            )
    }


    return (
        resultat,
        paire
    )


res_h1_125, H1_PAIRE_125 = (
    tester_h1(
        OOF_MT0_125_AGG,
        OOF_DUMMY_MEAN_125_AGG,
        125
    )
)


res_h1_105, H1_PAIRE_105 = (
    tester_h1(
        OOF_MT0_105_AGG,
        OOF_DUMMY_MEAN_105_AGG,
        105
    )
)


H1_MT0_VS_DUMMY = (
    pd.DataFrame(
        [
            res_h1_125,
            res_h1_105
        ]
    )
)


print(
    "=== TEST DE H1 : M_T0 RANDOM FOREST VS MOYENNE ==="
)


display(
    H1_MT0_VS_DUMMY.round(4)
)


H1_MT0_VS_DUMMY.to_csv(
    OUTPUT_DIR
    / "H1_MT0_vs_DummyMean.csv",
    index=False,
    encoding="utf-8-sig"
)


H1_PAIRE_125.to_csv(
    OUTPUT_DIR
    / "H1_erreurs_appariees_N125.csv",
    index=False,
    encoding="utf-8-sig"
)


H1_PAIRE_105.to_csv(
    OUTPUT_DIR
    / "H1_erreurs_appariees_N105.csv",
    index=False,
    encoding="utf-8-sig"
)

=== TEST DE H1 : M_T0 RANDOM FOREST VS MOYENNE ===


,COHORTE,N,MAE_DUMMY_MEAN,MAE_RANDOMFOREST_MT0,GAIN_MAE_RF_VS_DUMMY,IC95_BAS,IC95_HAUT,IC95_EXCLUT_0_POSITIVEMENT,STAT_WILCOXON,P_VALUE_UNILATERAL,H1_SOUTENUE
0,125,125,20.8607,20.1936,0.6671,-0.1846,1.5562,False,4607.0,0.0495,False
1,105,105,20.6987,20.3032,0.3955,-0.4810,1.2969,False,3087.0,0.1652,False


In [54]:
# ============================================================
# CELLULE 36 — Gain individuel sur prédictions OOF agrégées
# ============================================================

oof_paire_105 = (

    OOF_MT0_105_AGG[
        [
            "MODELE",
            "ROW_INDEX",
            "Y_TRUE",
            "Y_PRED",
            "ABS_ERROR"
        ]
    ]

    .merge(

        OOF_MT1_105_AGG[
            [
                "MODELE",
                "ROW_INDEX",
                "Y_TRUE",
                "Y_PRED",
                "ABS_ERROR"
            ]
        ],

        on=[
            "MODELE",
            "ROW_INDEX"
        ],

        suffixes=(
            "_MT0",
            "_MT1"
        ),

        validate="one_to_one"
    )
)


oof_paire_105[
    "GAIN_ABS_ERROR"
] = (
    oof_paire_105[
        "ABS_ERROR_MT0"
    ]
    -
    oof_paire_105[
        "ABS_ERROR_MT1"
    ]
)


resume_gain_individuel = (

    oof_paire_105

    .groupby(
        "MODELE",
        as_index=False
    )

    .agg(
        N=(
            "GAIN_ABS_ERROR",
            "count"
        ),

        GAIN_MAE_MOYEN=(
            "GAIN_ABS_ERROR",
            "mean"
        ),

        GAIN_MEDIAN=(
            "GAIN_ABS_ERROR",
            "median"
        ),

        PCT_APPRENANTS_AMELIORES=(
            "GAIN_ABS_ERROR",
            lambda x:
                (x > 0).mean() * 100
        )
    )
)


display(
    resume_gain_individuel
    .sort_values(
        "GAIN_MAE_MOYEN",
        ascending=False
    )
)

,MODELE,N,GAIN_MAE_MOYEN,GAIN_MEDIAN,PCT_APPRENANTS_AMELIORES
4,LinearRegression,105,1.802847e+11,2.021491,60.000000
0,DecisionTree,105,5.847827e+00,7.353905,69.523810
5,RandomForest,105,4.394131e+00,7.453189,67.619048
2,GradientBoosting,105,3.900014e+00,6.375617,68.571429
3,Lasso,105,2.776451e+00,3.368342,61.904762
6,Ridge,105,1.892012e+00,2.970653,64.761905
1,Dummy,105,-1.624098e-15,0.000000,5.714286


# Cellule 37 — Test apparié M_T0 vs M_T1

In [55]:
# ============================================================
# CELLULE 37 — Tests appariés + correction de Holm
# ============================================================

from scipy.stats import wilcoxon


# ------------------------------------------------------------
# 1. Vérifications préalables
# ------------------------------------------------------------

colonnes_requises = {
    "MODELE",
    "ABS_ERROR_MT0",
    "ABS_ERROR_MT1"
}


colonnes_manquantes = (
    colonnes_requises
    -
    set(
        oof_paire_105.columns
    )
)


assert len(
    colonnes_manquantes
) == 0, (
    "Colonnes manquantes dans oof_paire_105 : "
    f"{colonnes_manquantes}"
)


# ------------------------------------------------------------
# 2. Modèles à tester
#    Dummy exclu : il sert uniquement de référence minimale
# ------------------------------------------------------------

modeles_a_tester = sorted(
    [
        modele
        for modele in (
            oof_paire_105[
                "MODELE"
            ]
            .dropna()
            .unique()
        )
        if modele != "Dummy"
    ]
)


print(
    "Modèles testés :",
    modeles_a_tester
)


# ------------------------------------------------------------
# 3. Test de Wilcoxon apparié
# ------------------------------------------------------------

tests_apparies = []


for modele in modeles_a_tester:

    temp = (
        oof_paire_105[
            oof_paire_105[
                "MODELE"
            ] == modele
        ]
        .copy()
    )


    erreurs_mt0 = (
        temp[
            "ABS_ERROR_MT0"
        ]
        .astype(float)
        .to_numpy()
    )


    erreurs_mt1 = (
        temp[
            "ABS_ERROR_MT1"
        ]
        .astype(float)
        .to_numpy()
    )


    # Différence positive =
    # M_T1 produit une erreur plus faible
    gains = (
        erreurs_mt0
        -
        erreurs_mt1
    )


    # --------------------------------------------------------
    # Cas particulier :
    # si toutes les différences sont nulles,
    # Wilcoxon n'est pas informatif
    # --------------------------------------------------------

    if np.allclose(
        gains,
        0
    ):

        stat = 0.0
        p_value = 1.0

    else:

        stat, p_value = wilcoxon(
            erreurs_mt0,
            erreurs_mt1,
            alternative="two-sided"
        )


    tests_apparies.append({

        "MODELE":
            modele,

        "N":
            len(temp),

        "MAE_MT0":
            erreurs_mt0.mean(),

        "MAE_MT1":
            erreurs_mt1.mean(),

        "GAIN_MAE":
            gains.mean(),

        "GAIN_MEDIAN":
            np.median(
                gains
            ),

        "STAT_WILCOXON":
            stat,

        "P_VALUE":
            p_value
    })


tests_apparies = pd.DataFrame(
    tests_apparies
)


# ------------------------------------------------------------
# 4. Correction de Holm
# ------------------------------------------------------------

p_values = (
    tests_apparies[
        "P_VALUE"
    ]
    .to_numpy(
        dtype=float
    )
)


m = len(
    p_values
)


ordre = np.argsort(
    p_values
)


p_tries = (
    p_values[
        ordre
    ]
)


# Holm :
# p_i corrigé =
# max cumulatif de (m-i)*p_i
p_holm_tries = (
    np.maximum.accumulate(
        (
            m
            -
            np.arange(m)
        )
        *
        p_tries
    )
)


p_holm_tries = np.minimum(
    p_holm_tries,
    1.0
)


# Retour dans l'ordre initial
p_holm = np.empty(
    m,
    dtype=float
)


p_holm[
    ordre
] = p_holm_tries


tests_apparies[
    "P_VALUE_HOLM"
] = p_holm


tests_apparies[
    "SIGNIFICATIF_BRUT_005"
] = (
    tests_apparies[
        "P_VALUE"
    ] < 0.05
)


tests_apparies[
    "SIGNIFICATIF_HOLM_005"
] = (
    tests_apparies[
        "P_VALUE_HOLM"
    ] < 0.05
)


# ------------------------------------------------------------
# 5. Affichage
# ------------------------------------------------------------

tests_apparies = (
    tests_apparies
    .sort_values(
        "P_VALUE_HOLM"
    )
    .reset_index(
        drop=True
    )
)


display(
    tests_apparies
)


print(
    "\nNombre de comparaisons :",
    m
)

print(
    "Significatives après Holm :",
    int(
        tests_apparies[
            "SIGNIFICATIF_HOLM_005"
        ].sum()
    ),
    "/",
    m
)

Modèles testés : ['DecisionTree', 'GradientBoosting', 'Lasso', 'LinearRegression', 'RandomForest', 'Ridge']


,MODELE,N,MAE_MT0,MAE_MT1,GAIN_MAE,GAIN_MEDIAN,STAT_WILCOXON,P_VALUE,P_VALUE_HOLM,SIGNIFICATIF_BRUT_005,SIGNIFICATIF_HOLM_005
0,DecisionTree,105,2.130850e+01,15.460670,5.847827e+00,7.353905,1427.0,0.000015,0.000088,True,True
1,GradientBoosting,105,2.092414e+01,17.024128,3.900014e+00,6.375617,1560.0,0.000093,0.000465,True,True
2,RandomForest,105,2.030323e+01,15.909103,4.394131e+00,7.453189,1564.0,0.000098,0.000465,True,True
3,Lasso,105,2.007297e+01,17.296520,2.776451e+00,3.368342,1735.0,0.000812,0.002436,True,True
4,Ridge,105,1.996402e+01,18.072006,1.892012e+00,2.970653,1777.0,0.001307,0.002614,True,True
5,LinearRegression,105,1.802847e+11,25.907616,1.802847e+11,2.021491,1964.0,0.008881,0.008881,True,True



Nombre de comparaisons : 6
Significatives après Holm : 6 / 6


# Cellule 38 — Intervalle de confiance bootstrap du gain
C’est encore plus utile que le seul p-value.

In [56]:
# ============================================================
# CELLULE 38 — Bootstrap du gain MAE
# ============================================================

RNG = np.random.default_rng(
    RANDOM_STATE
)


N_BOOTSTRAP = 5000


bootstrap_resultats = []


for modele in [
    "DecisionTree",
    "RandomForest",
    "GradientBoosting",
    "Lasso",
    "Ridge"
]:

    temp = (
        oof_paire_105[
            oof_paire_105["MODELE"] == modele
        ]
        .copy()
    )


    gains = (
        temp["ABS_ERROR_MT0"]
        -
        temp["ABS_ERROR_MT1"]
    ).to_numpy()


    n = len(gains)


    gains_boot = np.empty(
        N_BOOTSTRAP
    )


    for b in range(
        N_BOOTSTRAP
    ):

        indices = RNG.integers(
            0,
            n,
            size=n
        )

        gains_boot[b] = (
            gains[indices]
            .mean()
        )


    ci_low = np.percentile(
        gains_boot,
        2.5
    )

    ci_high = np.percentile(
        gains_boot,
        97.5
    )


    bootstrap_resultats.append({

        "MODELE":
            modele,

        "GAIN_MAE_MOYEN":
            gains.mean(),

        "IC95_BAS":
            ci_low,

        "IC95_HAUT":
            ci_high,

        "IC_EXCLUT_0":
            (
                ci_low > 0
                or ci_high < 0
            )

    })


bootstrap_resultats = pd.DataFrame(
    bootstrap_resultats
)


display(
    bootstrap_resultats
    .sort_values(
        "GAIN_MAE_MOYEN",
        ascending=False
    )
)

,MODELE,GAIN_MAE_MOYEN,IC95_BAS,IC95_HAUT,IC_EXCLUT_0
0,DecisionTree,5.847827,3.408964,8.280378,True
1,RandomForest,4.394131,2.263618,6.471384,True
2,GradientBoosting,3.900014,1.976157,5.726440,True
3,Lasso,2.776451,1.124641,4.340028,True
4,Ridge,1.892012,0.662131,3.125263,True


#  Cellule 39 — Comparaison synthétique des candidats

In [57]:
# ============================================================
# CELLULE 39 — Synthèse des modèles candidats M_T1
# ============================================================

synthese_candidats = (
    resume_105[
        resume_105["FRAMEWORK"] == "M_T1"
    ]
    .merge(
        resume_gain_individuel[
            [
                "MODELE",
                "GAIN_MAE_MOYEN",
                "GAIN_MEDIAN",
                "PCT_APPRENANTS_AMELIORES"
            ]
        ],
        on="MODELE",
        how="left"
    )
    .merge(
    bootstrap_resultats[
        [
            "MODELE",
            "IC95_BAS",
            "IC95_HAUT",
            "IC_EXCLUT_0"
        ]
    ],
    on="MODELE",
    how="left",
    validate="one_to_one"
    )
)


display(
    synthese_candidats[
        synthese_candidats[
            "MODELE"
        ] != "Dummy"
    ]
    .sort_values(
        "MAE_MOYEN"
    )
)

,FRAMEWORK,MODELE,MAE_MOYEN,MAE_STD,RMSE_MOYEN,RMSE_STD,R2_MOYEN,R2_STD,GAIN_MAE_MOYEN,GAIN_MEDIAN,PCT_APPRENANTS_AMELIORES,IC95_BAS,IC95_HAUT,IC_EXCLUT_0
0,M_T1,DecisionTree,15.880854,2.654825,20.453543,2.962868,0.192167,0.189283,5.847827e+00,7.353905,69.523810,3.408964,8.280378,True
5,M_T1,RandomForest,16.219350,2.221532,19.995289,2.707394,0.233250,0.149917,4.394131e+00,7.453189,67.619048,2.263618,6.471384,True
2,M_T1,GradientBoosting,17.644016,2.522628,21.605545,3.238311,0.102674,0.210891,3.900014e+00,6.375617,68.571429,1.976157,5.726440,True
3,M_T1,Lasso,17.665365,2.875141,21.840006,3.511244,0.081576,0.266210,2.776451e+00,3.368342,61.904762,1.124641,4.340028,True
6,M_T1,Ridge,18.815422,2.634401,22.947769,3.386639,-0.016319,0.280025,1.892012e+00,2.970653,64.761905,0.662131,3.125263,True
4,M_T1,LinearRegression,29.562633,4.261072,36.731504,5.383603,-1.714955,1.063558,1.802847e+11,2.021491,60.000000,NaN,NaN,NaN


In [58]:
# ============================================================
# CELLULE 39B — Reconstruction des tables de résultats
# ============================================================

# -------------------------
# N = 125
# -------------------------

resultats_125 = pd.concat(
    [
        RESULTATS_MT0_125,
        RESULTATS_MT1_125
    ],
    ignore_index=True
)


resume_125 = (
    resultats_125
    .groupby(
        ["FRAMEWORK", "MODELE"],
        as_index=False
    )
    .agg(
        MAE_MOYEN=("MAE", "mean"),
        MAE_STD=("MAE", "std"),
        RMSE_MOYEN=("RMSE", "mean"),
        RMSE_STD=("RMSE", "std"),
        R2_MOYEN=("R2", "mean"),
        R2_STD=("R2", "std")
    )
)


comparaison_125 = (
    RESULTATS_MT0_125[
        ["MODELE", "FOLD", "MAE", "RMSE", "R2"]
    ]
    .merge(
        RESULTATS_MT1_125[
            ["MODELE", "FOLD", "MAE", "RMSE", "R2"]
        ],
        on=["MODELE", "FOLD"],
        suffixes=("_MT0", "_MT1"),
        validate="one_to_one"
    )
)


comparaison_125["DELTA_MAE"] = (
    comparaison_125["MAE_MT1"]
    - comparaison_125["MAE_MT0"]
)

comparaison_125["DELTA_RMSE"] = (
    comparaison_125["RMSE_MT1"]
    - comparaison_125["RMSE_MT0"]
)

comparaison_125["DELTA_R2"] = (
    comparaison_125["R2_MT1"]
    - comparaison_125["R2_MT0"]
)


resume_comparaison_125 = (
    comparaison_125
    .groupby("MODELE", as_index=False)
    .agg(
        DELTA_MAE_MOYEN=("DELTA_MAE", "mean"),
        DELTA_RMSE_MOYEN=("DELTA_RMSE", "mean"),
        DELTA_R2_MOYEN=("DELTA_R2", "mean")
    )
)


# -------------------------
# N = 105
# -------------------------

resultats_105 = pd.concat(
    [
        RESULTATS_MT0_105,
        RESULTATS_MT1_105
    ],
    ignore_index=True
)


resume_105 = (
    resultats_105
    .groupby(
        ["FRAMEWORK", "MODELE"],
        as_index=False
    )
    .agg(
        MAE_MOYEN=("MAE", "mean"),
        MAE_STD=("MAE", "std"),
        RMSE_MOYEN=("RMSE", "mean"),
        RMSE_STD=("RMSE", "std"),
        R2_MOYEN=("R2", "mean"),
        R2_STD=("R2", "std")
    )
)


comparaison_105 = (
    RESULTATS_MT0_105[
        ["MODELE", "FOLD", "MAE", "RMSE", "R2"]
    ]
    .merge(
        RESULTATS_MT1_105[
            ["MODELE", "FOLD", "MAE", "RMSE", "R2"]
        ],
        on=["MODELE", "FOLD"],
        suffixes=("_MT0", "_MT1"),
        validate="one_to_one"
    )
)


comparaison_105["DELTA_MAE"] = (
    comparaison_105["MAE_MT1"]
    - comparaison_105["MAE_MT0"]
)

comparaison_105["DELTA_RMSE"] = (
    comparaison_105["RMSE_MT1"]
    - comparaison_105["RMSE_MT0"]
)

comparaison_105["DELTA_R2"] = (
    comparaison_105["R2_MT1"]
    - comparaison_105["R2_MT0"]
)


resume_comparaison_105 = (
    comparaison_105
    .groupby("MODELE", as_index=False)
    .agg(
        DELTA_MAE_MOYEN=("DELTA_MAE", "mean"),
        DELTA_RMSE_MOYEN=("DELTA_RMSE", "mean"),
        DELTA_R2_MOYEN=("DELTA_R2", "mean")
    )
)


print("✅ Tables reconstruites")
print("resultats_125 :", resultats_125.shape)
print("resultats_105 :", resultats_105.shape)

display(resume_125)
display(resume_105)

✅ Tables reconstruites
resultats_125 : (700, 9)
resultats_105 : (700, 9)


,FRAMEWORK,MODELE,MAE_MOYEN,MAE_STD,RMSE_MOYEN,RMSE_STD,R2_MOYEN,R2_STD
0,M_T0,DecisionTree,2.104136e+01,2.690799e+00,2.504974e+01,3.250117e+00,-1.929486e-01,2.793783e-01
1,M_T0,Dummy,2.039131e+01,2.095964e+00,2.483064e+01,2.411839e+00,-1.637425e-01,1.637384e-01
2,M_T0,GradientBoosting,2.075058e+01,2.287856e+00,2.447627e+01,2.692650e+00,-1.371652e-01,2.356849e-01
3,M_T0,Lasso,2.080670e+01,2.028263e+00,2.431578e+01,2.209135e+00,-1.221836e-01,2.023908e-01
4,M_T0,LinearRegression,2.264795e+12,1.095369e+13,8.625201e+12,3.913807e+13,-2.859402e+24,1.397587e+25
5,M_T0,RandomForest,2.030644e+01,1.913631e+00,2.355044e+01,2.346691e+00,-4.669917e-02,1.547844e-01
6,M_T0,Ridge,2.053869e+01,2.008298e+00,2.398968e+01,2.561115e+00,-9.745999e-02,2.594519e-01
7,M_T1,DecisionTree,1.764204e+01,2.112502e+00,2.231746e+01,2.232147e+00,3.455227e-02,2.818580e-01
8,M_T1,Dummy,2.039131e+01,2.095964e+00,2.483064e+01,2.411839e+00,-1.637425e-01,1.637384e-01
9,M_T1,GradientBoosting,1.820336e+01,1.701584e+00,2.231020e+01,2.204640e+00,4.754532e-02,2.443165e-01


,FRAMEWORK,MODELE,MAE_MOYEN,MAE_STD,RMSE_MOYEN,RMSE_STD,R2_MOYEN,R2_STD
0,M_T0,DecisionTree,2.214683e+01,3.383974e+00,2.660336e+01,4.172194e+00,-3.583451e-01,3.245449e-01
1,M_T0,Dummy,1.973878e+01,3.139220e+00,2.426286e+01,3.894017e+00,-1.075279e-01,1.135031e-01
2,M_T0,GradientBoosting,2.142603e+01,2.707691e+00,2.539449e+01,3.906072e+00,-2.340192e-01,2.671825e-01
3,M_T0,Lasso,2.034681e+01,2.923246e+00,2.393439e+01,3.432208e+00,-9.939128e-02,2.481145e-01
4,M_T0,LinearRegression,1.802847e+11,1.274805e+12,8.042146e+11,5.686656e+12,-9.188842e+22,6.497492e+23
5,M_T0,RandomForest,2.040223e+01,2.529690e+00,2.379471e+01,3.390123e+00,-7.807058e-02,1.854788e-01
6,M_T0,Ridge,2.038374e+01,2.654246e+00,2.418001e+01,3.227049e+00,-1.256778e-01,2.574259e-01
7,M_T1,DecisionTree,1.588085e+01,2.654825e+00,2.045354e+01,2.962868e+00,1.921672e-01,1.892827e-01
8,M_T1,Dummy,1.973878e+01,3.139220e+00,2.426286e+01,3.894017e+00,-1.075279e-01,1.135031e-01
9,M_T1,GradientBoosting,1.764402e+01,2.522628e+00,2.160554e+01,3.238311e+00,1.026738e-01,2.108911e-01


Cellule 40 — Export des résultats du notebook 03

In [59]:
# ============================================================
# CELLULE 40 — Exports
# ============================================================

exports_03 = {

    "resultats_nested_cv_N125.csv":
        resultats_125,

    "comparaison_MT0_MT1_N125.csv":
        comparaison_125,

    "resume_comparaison_N125.csv":
        resume_comparaison_125,

    "resultats_nested_cv_N105.csv":
        resultats_105,

    "resume_modeles_N105.csv":
        resume_105,

    "comparaison_MT0_MT1_N105.csv":
        comparaison_105,

    "resume_comparaison_N105.csv":
        resume_comparaison_105,

    "oof_MT0_N105.csv":
        OOF_MT0_105,

    "oof_MT1_N105.csv":
        OOF_MT1_105,

    "gain_individuel_N105.csv":
        oof_paire_105,

    "resume_gain_individuel_N105.csv":
        resume_gain_individuel,

    "tests_wilcoxon_N105.csv":
        tests_apparies,

    "bootstrap_gain_N105.csv":
        bootstrap_resultats,

    "oof_MT0_N125_agrege.csv":
        OOF_MT0_125_AGG,

    "oof_MT1_N125_agrege.csv":
        OOF_MT1_125_AGG,

    "oof_MT0_N105_agrege.csv":
        OOF_MT0_105_AGG,

    "oof_MT1_N105_agrege.csv":
        OOF_MT1_105_AGG,

    "T1_scoring_long.csv":
        T1_SCORING_LONG,

    "T1_scores.csv":
        T1_SCORES,
    
    "synthese_candidats_M_T1.csv":
        synthese_candidats,

    "parametres_MT0_N125.csv":
        PARAMS_MT0_125,

    "parametres_MT1_N125.csv":
        PARAMS_MT1_125,

    "parametres_MT0_N105.csv":
        PARAMS_MT0_105,

    "parametres_MT1_N105.csv":
        PARAMS_MT1_105
}


for nom, df in exports_03.items():

    df.to_csv(
        OUTPUT_DIR / nom,
        index=False,
        encoding="utf-8-sig"
    )


print(
    f"✅ {len(exports_03)} fichiers exportés"
)

✅ 24 fichiers exportés


In [60]:
# ============================================================
# CELLULE 41 — Modèles avancés pour petit dataset tabulaire
# ============================================================

from sklearn.model_selection import RandomizedSearchCV

from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)

from sklearn.svm import SVR


MODELES_AVANCES = {

    # --------------------------------------------------------
    # Decision Tree optimisé
    # --------------------------------------------------------
    "DecisionTree_OPT": {

        "model": DecisionTreeRegressor(
            random_state=RANDOM_STATE
        ),

        "params": {
            "model__max_depth":
                [2, 3, 4, 5, 6, 8, None],

            "model__min_samples_leaf":
                [2, 3, 4, 5, 8, 10, 15],

            "model__min_samples_split":
                [4, 6, 8, 10, 15, 20],

            "model__max_features":
                [None, "sqrt", 0.7, 0.9]
        }
    },


    # --------------------------------------------------------
    # Random Forest optimisé
    # --------------------------------------------------------
    "RandomForest_OPT": {

        "model": RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1
        ),

        "params": {
            "model__n_estimators":
                [300, 500, 800, 1000],

            "model__max_depth":
                [3, 4, 5, 6, 8, 10, None],

            "model__min_samples_leaf":
                [1, 2, 3, 4, 5, 8],

            "model__min_samples_split":
                [2, 4, 6, 10],

            "model__max_features":
                ["sqrt", 0.4, 0.6, 0.8, 1.0]
        }
    },


    # --------------------------------------------------------
    # Extra Trees
    # --------------------------------------------------------
    "ExtraTrees": {

        "model": ExtraTreesRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1
        ),

        "params": {
            "model__n_estimators":
                [300, 500, 800],

            "model__max_depth":
                [3, 4, 5, 6, 8, None],

            "model__min_samples_leaf":
                [1, 2, 3, 4, 5, 8],

            "model__max_features":
                ["sqrt", 0.5, 0.7, 1.0]
        }
    },


    # --------------------------------------------------------
    # Gradient Boosting optimisé
    # --------------------------------------------------------
    "GradientBoosting_OPT": {

        "model": GradientBoostingRegressor(
            random_state=RANDOM_STATE
        ),

        "params": {
            "model__n_estimators":
                [50, 75, 100, 150, 200, 300],

            "model__learning_rate":
                [0.01, 0.02, 0.03, 0.05, 0.1],

            "model__max_depth":
                [1, 2, 3],

            "model__min_samples_leaf":
                [2, 3, 5, 8, 10],

            "model__subsample":
                [0.6, 0.8, 1.0],

            "model__max_features":
                [None, "sqrt", 0.7]
        }
    },


    # --------------------------------------------------------
    # HistGradientBoosting
    # --------------------------------------------------------
    "HistGradientBoosting": {

        "model": HistGradientBoostingRegressor(
            random_state=RANDOM_STATE
        ),

        "params": {
            "model__learning_rate":
                [0.02, 0.03, 0.05, 0.1],

            "model__max_iter":
                [100, 200, 300, 500],

            "model__max_leaf_nodes":
                [5, 7, 10, 15, 20],

            "model__min_samples_leaf":
                [5, 10, 15, 20],

            "model__l2_regularization":
                [0.0, 0.1, 1.0, 5.0, 10.0]
        }
    },


    # --------------------------------------------------------
    # SVR RBF
    # Très intéressant sur petits échantillons
    # --------------------------------------------------------
    "SVR_RBF": {

        "model": SVR(
            kernel="rbf"
        ),

        "params": {
            "model__C":
                [0.5, 1, 2, 5, 10, 20, 50, 100],

            "model__epsilon":
                [0.1, 0.5, 1, 2, 5, 10],

            "model__gamma":
                ["scale", "auto", 0.001, 0.005, 0.01, 0.05, 0.1]
        }
    }
}


print(
    "Modèles avancés :",
    list(MODELES_AVANCES.keys())
)

Modèles avancés : ['DecisionTree_OPT', 'RandomForest_OPT', 'ExtraTrees', 'GradientBoosting_OPT', 'HistGradientBoosting', 'SVR_RBF']


In [61]:
# ============================================================
# CELLULE 42 — Nested CV avec optimisation avancée
# ============================================================

def evaluer_nested_cv_optimise(
    X,
    y,
    variables_num,
    variables_cat,
    outer_splits,
    framework,
    analyse,
    n_iter=25
):

    resultats = []
    predictions = []
    meilleurs_params = []


    for nom_modele, spec in MODELES_AVANCES.items():

        print(
            f"\n{'='*75}"
        )

        print(
            analyse,
            "|",
            framework,
            "|",
            nom_modele
        )

        print(
            "="*75
        )


        for fold, (
            train_idx,
            test_idx
        ) in enumerate(
            outer_splits,
            start=1
        ):

            X_train = X.iloc[
                train_idx
            ].copy()

            X_test = X.iloc[
                test_idx
            ].copy()

            y_train = y.iloc[
                train_idx
            ].copy()

            y_test = y.iloc[
                test_idx
            ].copy()


            preprocessor = creer_preprocesseur(
                variables_num,
                variables_cat
            )


            pipe = Pipeline([

                (
                    "preprocess",
                    preprocessor
                ),

                (
                    "model",
                    clone(
                        spec["model"]
                    )
                )

            ])


            inner_cv = KFold(
                n_splits=4,
                shuffle=True,
                random_state=(
                    RANDOM_STATE
                    + fold
                )
            )


            recherche = RandomizedSearchCV(

                estimator=pipe,

                param_distributions=
                    spec["params"],

                n_iter=n_iter,

                scoring=
                    "neg_mean_absolute_error",

                cv=inner_cv,

                random_state=(
                    RANDOM_STATE
                    + fold
                ),

                n_jobs=-1,

                refit=True,

                error_score="raise"
            )


            recherche.fit(
                X_train,
                y_train
            )


            pred = recherche.predict(
                X_test
            )


            mae = mean_absolute_error(
                y_test,
                pred
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y_test,
                    pred
                )
            )

            r2 = r2_score(
                y_test,
                pred
            )


            resultats.append({

                "ANALYSE":
                    analyse,

                "FRAMEWORK":
                    framework,

                "MODELE":
                    nom_modele,

                "FOLD":
                    fold,

                "MAE":
                    mae,

                "RMSE":
                    rmse,

                "R2":
                    r2
            })


            meilleurs_params.append({

                "ANALYSE":
                    analyse,

                "FRAMEWORK":
                    framework,

                "MODELE":
                    nom_modele,

                "FOLD":
                    fold,

                "BEST_PARAMS":
                    json.dumps(
                        recherche.best_params_,
                        ensure_ascii=False
                    )
            })


            for idx, vrai, estime in zip(
                test_idx,
                y_test,
                pred
            ):

                predictions.append({

                    "ANALYSE":
                        analyse,

                    "FRAMEWORK":
                        framework,

                    "MODELE":
                        nom_modele,

                    "FOLD":
                        fold,

                    "ROW_INDEX":
                        int(idx),

                    "Y_TRUE":
                        float(vrai),

                    "Y_PRED":
                        float(estime),

                    "ABS_ERROR":
                        float(
                            abs(
                                estime - vrai
                            )
                        )
                })


            print(
                f"Fold {fold} | "
                f"MAE={mae:.3f} | "
                f"RMSE={rmse:.3f} | "
                f"R²={r2:.3f}"
            )


    return (
        pd.DataFrame(resultats),
        pd.DataFrame(predictions),
        pd.DataFrame(meilleurs_params)
    )

Cellule 43 — Tester d’abord M_T1 sur N=125

In [62]:
# ============================================================
# CELLULE 43 — Modèles avancés M_T1, N=125
# ============================================================

ADV_MT1_125, ADV_OOF_MT1_125, ADV_PARAMS_MT1_125 = (
    evaluer_nested_cv_optimise(

        X=X1_125,
        y=y_125,

        variables_num=
            FEATURES_MT1_NUM,

        variables_cat=
            FEATURES_MT1_CAT,

        outer_splits=
            OUTER_SPLITS_125_EXPLO,

        framework=
            "M_T1",

        analyse=
            "EXPLORATOIRE_N125",

        n_iter=10
    )
)


EXPLORATOIRE_N125 | M_T1 | DecisionTree_OPT
Fold 1 | MAE=18.682 | RMSE=23.802 | R²=-0.048
Fold 2 | MAE=13.358 | RMSE=16.386 | R²=0.456
Fold 3 | MAE=15.781 | RMSE=21.418 | R²=0.202
Fold 4 | MAE=18.120 | RMSE=23.157 | R²=-0.331
Fold 5 | MAE=19.327 | RMSE=24.021 | R²=0.173

EXPLORATOIRE_N125 | M_T1 | RandomForest_OPT
Fold 1 | MAE=17.865 | RMSE=23.564 | R²=-0.027
Fold 2 | MAE=14.555 | RMSE=16.832 | R²=0.426
Fold 3 | MAE=16.437 | RMSE=21.075 | R²=0.228
Fold 4 | MAE=14.916 | RMSE=17.660 | R²=0.226
Fold 5 | MAE=18.937 | RMSE=23.367 | R²=0.218

EXPLORATOIRE_N125 | M_T1 | ExtraTrees
Fold 1 | MAE=18.941 | RMSE=22.816 | R²=0.037
Fold 2 | MAE=15.912 | RMSE=17.841 | R²=0.355
Fold 3 | MAE=18.719 | RMSE=22.770 | R²=0.098
Fold 4 | MAE=15.512 | RMSE=18.566 | R²=0.144
Fold 5 | MAE=19.851 | RMSE=24.362 | R²=0.150

EXPLORATOIRE_N125 | M_T1 | GradientBoosting_OPT
Fold 1 | MAE=18.258 | RMSE=21.923 | R²=0.111
Fold 2 | MAE=15.652 | RMSE=17.622 | R²=0.371
Fold 3 | MAE=19.997 | RMSE=24.381 | R²=-0.034
Fold 4 |

Cellule 44 — Synthèse N=125

In [63]:
# ============================================================
# CELLULE 44 — Résultats optimisés N=125
# ============================================================

resume_adv_125 = (
    ADV_MT1_125

    .groupby(
        "MODELE",
        as_index=False
    )

    .agg(

        MAE_MOYEN=(
            "MAE",
            "mean"
        ),

        MAE_STD=(
            "MAE",
            "std"
        ),

        RMSE_MOYEN=(
            "RMSE",
            "mean"
        ),

        RMSE_STD=(
            "RMSE",
            "std"
        ),

        R2_MOYEN=(
            "R2",
            "mean"
        ),

        R2_STD=(
            "R2",
            "std"
        )

    )
)


display(
    resume_adv_125
    .sort_values(
        "MAE_MOYEN"
    )
)

,MODELE,MAE_MOYEN,MAE_STD,RMSE_MOYEN,RMSE_STD,R2_MOYEN,R2_STD
3,HistGradientBoosting,16.470698,1.731752,20.882508,2.686097,0.184097,0.140622
4,RandomForest_OPT,16.542174,1.876758,20.499819,3.140886,0.213952,0.160993
0,DecisionTree_OPT,17.053597,2.461448,21.756911,3.171224,0.090426,0.295960
2,GradientBoosting_OPT,17.352181,2.276457,20.809335,3.284939,0.192679,0.157342
1,ExtraTrees,17.787004,1.946570,21.271148,2.883935,0.156873,0.119859
5,SVR_RBF,17.882137,1.944061,21.617602,2.988468,0.132424,0.098088


Cellule 45 — Tester les mêmes modèles sur N=105

In [64]:
# ============================================================
# CELLULE 45 — Modèles avancés M_T1, N=105
# ============================================================

ADV_MT1_105, ADV_OOF_MT1_105, ADV_PARAMS_MT1_105 = (
    evaluer_nested_cv_optimise(

        X=X1_105,
        y=y_105,

        variables_num=
            FEATURES_MT1_NUM,

        variables_cat=
            FEATURES_MT1_CAT,

        outer_splits=
            OUTER_SPLITS_105_EXPLO,

        framework=
            "M_T1",

        analyse=
            "EXPLORATOIRE_N105",

        n_iter=10
    )
)


EXPLORATOIRE_N105 | M_T1 | DecisionTree_OPT
Fold 1 | MAE=13.899 | RMSE=17.739 | R²=0.246
Fold 2 | MAE=14.171 | RMSE=17.934 | R²=0.294
Fold 3 | MAE=14.562 | RMSE=20.733 | R²=0.355
Fold 4 | MAE=18.499 | RMSE=24.615 | R²=0.043
Fold 5 | MAE=16.694 | RMSE=20.703 | R²=0.301

EXPLORATOIRE_N105 | M_T1 | RandomForest_OPT
Fold 1 | MAE=15.268 | RMSE=17.791 | R²=0.242
Fold 2 | MAE=14.129 | RMSE=17.472 | R²=0.330
Fold 3 | MAE=15.795 | RMSE=20.167 | R²=0.390
Fold 4 | MAE=17.249 | RMSE=22.497 | R²=0.201
Fold 5 | MAE=18.360 | RMSE=22.101 | R²=0.204

EXPLORATOIRE_N105 | M_T1 | ExtraTrees
Fold 1 | MAE=15.035 | RMSE=17.744 | R²=0.246
Fold 2 | MAE=15.788 | RMSE=18.053 | R²=0.285
Fold 3 | MAE=17.586 | RMSE=21.744 | R²=0.291
Fold 4 | MAE=18.156 | RMSE=23.545 | R²=0.124
Fold 5 | MAE=20.863 | RMSE=24.280 | R²=0.039

EXPLORATOIRE_N105 | M_T1 | GradientBoosting_OPT
Fold 1 | MAE=15.137 | RMSE=18.449 | R²=0.185
Fold 2 | MAE=14.817 | RMSE=17.768 | R²=0.307
Fold 3 | MAE=18.153 | RMSE=21.749 | R²=0.290
Fold 4 | MAE

# Cellule 46 — Synthèse N=105

In [65]:
# ============================================================
# CELLULE 46 — Résultats avancés N=105
# ============================================================

resume_adv_105 = (
    ADV_MT1_105

    .groupby(
        "MODELE",
        as_index=False
    )

    .agg(

        MAE_MOYEN=(
            "MAE",
            "mean"
        ),

        MAE_STD=(
            "MAE",
            "std"
        ),

        RMSE_MOYEN=(
            "RMSE",
            "mean"
        ),

        RMSE_STD=(
            "RMSE",
            "std"
        ),

        R2_MOYEN=(
            "R2",
            "mean"
        ),

        R2_STD=(
            "R2",
            "std"
        )

    )
)


display(
    resume_adv_105
    .sort_values(
        "MAE_MOYEN"
    )
)

,MODELE,MAE_MOYEN,MAE_STD,RMSE_MOYEN,RMSE_STD,R2_MOYEN,R2_STD
0,DecisionTree_OPT,15.564897,1.975393,20.344867,2.788877,0.247878,0.120872
4,RandomForest_OPT,16.160341,1.664452,20.005724,2.342519,0.273127,0.083534
3,HistGradientBoosting,16.335504,1.774349,19.902930,2.999491,0.283063,0.110831
2,GradientBoosting_OPT,16.648406,1.633882,20.366959,2.078576,0.247466,0.052208
1,ExtraTrees,17.485448,2.277907,21.073138,3.043412,0.196850,0.110790
5,SVR_RBF,18.176985,2.166398,22.465167,3.027471,0.084615,0.129340


Cellule 47 — Vérifier CatBoost

In [66]:
# ============================================================
# CELLULE 47 — CatBoost
# ============================================================

try:

    from catboost import (
        CatBoostRegressor
    )

    CATBOOST_OK = True

    print(
        "✅ CatBoost disponible"
    )

except ImportError:

    CATBOOST_OK = False

    print(
        "❌ CatBoost non installé"
    )

    print(
        "Installer avec : pip install catboost"
    )

✅ CatBoost disponible


# Cellule 48 — Préparation CatBoost
Ici pas de OneHotEncoder.

In [67]:
# ============================================================
# CELLULE 48 — Données natives pour CatBoost
# ============================================================

def preparer_catboost(
    X,
    variables_cat
):

    X_cb = X.copy()


    for col in variables_cat:

        X_cb[col] = (
            X_cb[col]
            .fillna("__MANQUANT__")
            .astype(str)
        )


    return X_cb


XCB_125 = preparer_catboost(
    X1_125,
    FEATURES_MT1_CAT
)


XCB_105 = preparer_catboost(
    X1_105,
    FEATURES_MT1_CAT
)


print(
    XCB_125.shape,
    XCB_105.shape
)

(125, 30) (105, 30)


#  Cellule 49 — Nested CV CatBoost

In [68]:
# ============================================================
# CELLULE 49 — Nested CV CatBoost exploratoire
# ============================================================

def evaluer_catboost(
    X,
    y,
    outer_splits,
    analyse,
    n_iter=10
):

    resultats = []
    oof = []
    params_resultats = []


    grille = {

        "depth": [
            2,
            3,
            4,
            5,
            6
        ],

        "learning_rate": [
            0.01,
            0.03,
            0.05,
            0.1
        ],

        "iterations": [
            150,
            250,
            400,
            600
        ],

        "l2_leaf_reg": [
            1,
            3,
            5,
            10,
            20,
            50
        ],

        "random_strength": [
            0,
            0.5,
            1,
            2
        ]
    }


    for fold, (
        train_idx,
        test_idx
    ) in enumerate(
        outer_splits,
        start=1
    ):

        print(
            f"\n{analyse} — Fold "
            f"{fold}/{len(outer_splits)}"
        )


        X_train = (
            X.iloc[
                train_idx
            ]
            .copy()
        )

        X_test = (
            X.iloc[
                test_idx
            ]
            .copy()
        )

        y_train = (
            y.iloc[
                train_idx
            ]
            .copy()
        )

        y_test = (
            y.iloc[
                test_idx
            ]
            .copy()
        )


        modele = CatBoostRegressor(

            loss_function="RMSE",

            random_seed=RANDOM_STATE,

            verbose=0,

            allow_writing_files=False,

            # RandomizedSearchCV gère déjà
            # le parallélisme externe
            thread_count=1,

            cat_features=
                FEATURES_MT1_CAT
        )


        inner_cv = KFold(
            n_splits=4,
            shuffle=True,
            random_state=
                RANDOM_STATE
                +
                fold
        )


        recherche = RandomizedSearchCV(

            estimator=
                modele,

            param_distributions=
                grille,

            n_iter=
                n_iter,

            scoring=
                "neg_mean_absolute_error",

            cv=
                inner_cv,

            random_state=
                RANDOM_STATE
                +
                fold,

            n_jobs=-1,

            refit=True,

            error_score="raise"
        )


        recherche.fit(
            X_train,
            y_train
        )


        pred = (
            recherche
            .predict(
                X_test
            )
        )


        mae = mean_absolute_error(
            y_test,
            pred
        )


        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                pred
            )
        )


        r2 = r2_score(
            y_test,
            pred
        )


        resultats.append({

            "ANALYSE":
                analyse,

            "FRAMEWORK":
                "M_T1",

            "MODELE":
                "CatBoost",

            "FOLD":
                fold,

            "MAE":
                mae,

            "RMSE":
                rmse,

            "R2":
                r2
        })


        params_resultats.append({

            "ANALYSE":
                analyse,

            "FRAMEWORK":
                "M_T1",

            "MODELE":
                "CatBoost",

            "FOLD":
                fold,

            "BEST_PARAMS":
                json.dumps(
                    recherche.best_params_,
                    ensure_ascii=False
                )
        })


        for idx, vrai, estime in zip(
            test_idx,
            y_test,
            pred
        ):

            oof.append({

                "ANALYSE":
                    analyse,

                "FRAMEWORK":
                    "M_T1",

                "MODELE":
                    "CatBoost",

                "FOLD":
                    fold,

                "ROW_INDEX":
                    int(idx),

                "Y_TRUE":
                    float(vrai),

                "Y_PRED":
                    float(estime),

                "ABS_ERROR":
                    float(
                        abs(
                            estime
                            -
                            vrai
                        )
                    )
            })


        print(
            f"MAE={mae:.3f} | "
            f"RMSE={rmse:.3f} | "
            f"R²={r2:.3f}"
        )


    return (

        pd.DataFrame(
            resultats
        ),

        pd.DataFrame(
            oof
        ),

        pd.DataFrame(
            params_resultats
        )
    )

In [69]:
CAT_125, CAT_OOF_125, CAT_PARAMS_125 = (
    evaluer_catboost(
        X=XCB_125,
        y=y_125,
        outer_splits=OUTER_SPLITS_125_EXPLO,
        analyse="CATBOOST_EXPLORATOIRE_N125",
        n_iter=10
    )
)


CAT_105, CAT_OOF_105, CAT_PARAMS_105 = (
    evaluer_catboost(
        X=XCB_105,
        y=y_105,
        outer_splits=OUTER_SPLITS_105_EXPLO,
        analyse="CATBOOST_EXPLORATOIRE_N105",
        n_iter=10
    )
)


CATBOOST_EXPLORATOIRE_N125 — Fold 1/5
MAE=18.819 | RMSE=22.773 | R²=0.040

CATBOOST_EXPLORATOIRE_N125 — Fold 2/5
MAE=15.061 | RMSE=18.127 | R²=0.335

CATBOOST_EXPLORATOIRE_N125 — Fold 3/5
MAE=19.932 | RMSE=24.216 | R²=-0.020

CATBOOST_EXPLORATOIRE_N125 — Fold 4/5
MAE=15.448 | RMSE=17.500 | R²=0.240

CATBOOST_EXPLORATOIRE_N125 — Fold 5/5
MAE=19.115 | RMSE=24.149 | R²=0.165

CATBOOST_EXPLORATOIRE_N105 — Fold 1/5
MAE=16.060 | RMSE=19.126 | R²=0.124

CATBOOST_EXPLORATOIRE_N105 — Fold 2/5
MAE=15.486 | RMSE=18.926 | R²=0.214

CATBOOST_EXPLORATOIRE_N105 — Fold 3/5
MAE=16.428 | RMSE=21.282 | R²=0.321

CATBOOST_EXPLORATOIRE_N105 — Fold 4/5
MAE=17.646 | RMSE=22.554 | R²=0.197

CATBOOST_EXPLORATOIRE_N105 — Fold 5/5
MAE=17.830 | RMSE=20.572 | R²=0.310


Cellule 50 — Comparaison finale de tous les candidats

In [70]:
# ============================================================
# CELLULE 50 — Comparaison finale des modèles optimisés
# ============================================================

comparaison_finale_125 = pd.concat(
    [
        ADV_MT1_125,
        CAT_125
    ],
    ignore_index=True
)


resume_final_125 = (
    comparaison_finale_125

    .groupby(
        "MODELE",
        as_index=False
    )

    .agg(

        MAE_MOYEN=("MAE", "mean"),
        MAE_STD=("MAE", "std"),

        RMSE_MOYEN=("RMSE", "mean"),
        RMSE_STD=("RMSE", "std"),

        R2_MOYEN=("R2", "mean"),
        R2_STD=("R2", "std")

    )

    .sort_values(
        "MAE_MOYEN"
    )
)


display(
    resume_final_125
)

,MODELE,MAE_MOYEN,MAE_STD,RMSE_MOYEN,RMSE_STD,R2_MOYEN,R2_STD
4,HistGradientBoosting,16.470698,1.731752,20.882508,2.686097,0.184097,0.140622
5,RandomForest_OPT,16.542174,1.876758,20.499819,3.140886,0.213952,0.160993
1,DecisionTree_OPT,17.053597,2.461448,21.756911,3.171224,0.090426,0.295960
3,GradientBoosting_OPT,17.352181,2.276457,20.809335,3.284939,0.192679,0.157342
0,CatBoost,17.674925,2.251080,21.353127,3.289468,0.151852,0.144184
2,ExtraTrees,17.787004,1.946570,21.271148,2.883935,0.156873,0.119859
6,SVR_RBF,17.882137,1.944061,21.617602,2.988468,0.132424,0.098088


Et pour N=105 :

In [71]:
comparaison_finale_105 = pd.concat(
    [
        ADV_MT1_105,
        CAT_105
    ],
    ignore_index=True
)


resume_final_105 = (
    comparaison_finale_105

    .groupby(
        "MODELE",
        as_index=False
    )

    .agg(

        MAE_MOYEN=("MAE", "mean"),
        MAE_STD=("MAE", "std"),

        RMSE_MOYEN=("RMSE", "mean"),
        RMSE_STD=("RMSE", "std"),

        R2_MOYEN=("R2", "mean"),
        R2_STD=("R2", "std")

    )

    .sort_values(
        "MAE_MOYEN"
    )
)


display(
    resume_final_105
)

,MODELE,MAE_MOYEN,MAE_STD,RMSE_MOYEN,RMSE_STD,R2_MOYEN,R2_STD
1,DecisionTree_OPT,15.564897,1.975393,20.344867,2.788877,0.247878,0.120872
5,RandomForest_OPT,16.160341,1.664452,20.005724,2.342519,0.273127,0.083534
4,HistGradientBoosting,16.335504,1.774349,19.902930,2.999491,0.283063,0.110831
3,GradientBoosting_OPT,16.648406,1.633882,20.366959,2.078576,0.247466,0.052208
0,CatBoost,16.690035,1.015569,20.492019,1.516607,0.232881,0.082527
2,ExtraTrees,17.485448,2.277907,21.073138,3.043412,0.196850,0.110790
6,SVR_RBF,18.176985,2.166398,22.465167,3.027471,0.084615,0.129340


In [72]:
# ============================================================
# CELLULE FINALE — Classement global des modèles
# Performance + stabilité N=125 / N=105
# ============================================================

# ------------------------------------------------------------
# 1. Modèles initiaux M_T1
# ------------------------------------------------------------

initial_125 = (
    resume_125[
        resume_125["FRAMEWORK"] == "M_T1"
    ]
    .drop(columns=["FRAMEWORK"])
    .copy()
)

initial_105 = (
    resume_105[
        resume_105["FRAMEWORK"] == "M_T1"
    ]
    .drop(columns=["FRAMEWORK"])
    .copy()
)


# ------------------------------------------------------------
# 2. Modèles avancés / optimisés
# ------------------------------------------------------------

avance_125 = resume_final_125.copy()
avance_105 = resume_final_105.copy()


# ------------------------------------------------------------
# 3. Tous les modèles testés
# ------------------------------------------------------------

tous_125 = pd.concat(
    [
        initial_125,
        avance_125
    ],
    ignore_index=True
).drop_duplicates(
    subset="MODELE",
    keep="last"
)


tous_105 = pd.concat(
    [
        initial_105,
        avance_105
    ],
    ignore_index=True
).drop_duplicates(
    subset="MODELE",
    keep="last"
)


# ------------------------------------------------------------
# 4. Fusion N=125 et N=105
# ------------------------------------------------------------

classement_final = tous_125.merge(
    tous_105,
    on="MODELE",
    suffixes=("_125", "_105"),
    how="inner"
)


# ------------------------------------------------------------
# 5. Performance moyenne entre les deux analyses
# ------------------------------------------------------------

classement_final["MAE_GLOBAL"] = (
    classement_final["MAE_MOYEN_125"]
    +
    classement_final["MAE_MOYEN_105"]
) / 2


classement_final["RMSE_GLOBAL"] = (
    classement_final["RMSE_MOYEN_125"]
    +
    classement_final["RMSE_MOYEN_105"]
) / 2


classement_final["R2_GLOBAL"] = (
    classement_final["R2_MOYEN_125"]
    +
    classement_final["R2_MOYEN_105"]
) / 2


# ------------------------------------------------------------
# 6. Stabilité entre folds
# ------------------------------------------------------------

classement_final["R2_STD_GLOBAL"] = (
    classement_final["R2_STD_125"]
    +
    classement_final["R2_STD_105"]
) / 2


classement_final["MAE_STD_GLOBAL"] = (
    classement_final["MAE_STD_125"]
    +
    classement_final["MAE_STD_105"]
) / 2


# ------------------------------------------------------------
# 7. Stabilité entre N=125 et N=105
# ------------------------------------------------------------

classement_final["ECART_MAE_COHORTES"] = abs(
    classement_final["MAE_MOYEN_125"]
    -
    classement_final["MAE_MOYEN_105"]
)


classement_final["ECART_R2_COHORTES"] = abs(
    classement_final["R2_MOYEN_125"]
    -
    classement_final["R2_MOYEN_105"]
)


# ------------------------------------------------------------
# 8. Rangs de performance
# ------------------------------------------------------------

classement_final["RANG_MAE"] = (
    classement_final["MAE_GLOBAL"]
    .rank(method="min", ascending=True)
)


classement_final["RANG_RMSE"] = (
    classement_final["RMSE_GLOBAL"]
    .rank(method="min", ascending=True)
)


classement_final["RANG_R2"] = (
    classement_final["R2_GLOBAL"]
    .rank(method="min", ascending=False)
)


classement_final["RANG_PERFORMANCE"] = (
    classement_final[
        [
            "RANG_MAE",
            "RANG_RMSE",
            "RANG_R2"
        ]
    ]
    .mean(axis=1)
)


# ------------------------------------------------------------
# 9. Rangs de stabilité / robustesse
# ------------------------------------------------------------

classement_final["RANG_R2_STD"] = (
    classement_final["R2_STD_GLOBAL"]
    .rank(method="min", ascending=True)
)


classement_final["RANG_MAE_STD"] = (
    classement_final["MAE_STD_GLOBAL"]
    .rank(method="min", ascending=True)
)


classement_final["RANG_ECART_MAE"] = (
    classement_final["ECART_MAE_COHORTES"]
    .rank(method="min", ascending=True)
)


classement_final["RANG_ECART_R2"] = (
    classement_final["ECART_R2_COHORTES"]
    .rank(method="min", ascending=True)
)


classement_final["RANG_STABILITE"] = (
    classement_final[
        [
            "RANG_R2_STD",
            "RANG_MAE_STD",
            "RANG_ECART_MAE",
            "RANG_ECART_R2"
        ]
    ]
    .mean(axis=1)
)


# ------------------------------------------------------------
# 10. Rang global
#     60 % performance
#     40 % stabilité
# ------------------------------------------------------------

classement_final["SCORE_GLOBAL"] = (
    0.60 * classement_final["RANG_PERFORMANCE"]
    +
    0.40 * classement_final["RANG_STABILITE"]
)


classement_final["RANG_GLOBAL"] = (
    classement_final["SCORE_GLOBAL"]
    .rank(method="min", ascending=True)
)


# ------------------------------------------------------------
# 11. Tableau final
# ------------------------------------------------------------

colonnes_finales = [
    "MODELE",

    "MAE_MOYEN_125",
    "MAE_MOYEN_105",
    "MAE_GLOBAL",

    "RMSE_GLOBAL",

    "R2_MOYEN_125",
    "R2_MOYEN_105",
    "R2_GLOBAL",

    "R2_STD_GLOBAL",

    "ECART_MAE_COHORTES",
    "ECART_R2_COHORTES",

    "RANG_PERFORMANCE",
    "RANG_STABILITE",
    "SCORE_GLOBAL",
    "RANG_GLOBAL"
]


classement_final = (
    classement_final[
        colonnes_finales
    ]
    .sort_values(
        [
            "RANG_GLOBAL",
            "MAE_GLOBAL"
        ]
    )
    .reset_index(drop=True)
)


display(
    classement_final
)


# ------------------------------------------------------------
# 12. Meilleurs modèles selon chaque critère
# ------------------------------------------------------------

best_mae = classement_final.loc[
    classement_final["MAE_GLOBAL"].idxmin()
]

best_rmse = classement_final.loc[
    classement_final["RMSE_GLOBAL"].idxmin()
]

best_r2 = classement_final.loc[
    classement_final["R2_GLOBAL"].idxmax()
]

best_stable = classement_final.loc[
    classement_final["RANG_STABILITE"].idxmin()
]

best_global = classement_final.loc[
    classement_final["RANG_GLOBAL"].idxmin()
]


print("\n==============================================")
print("        SYNTHÈSE FINALE ÉTAPE 03")
print("==============================================")

print(
    f"Meilleur MAE        : "
    f"{best_mae['MODELE']} "
    f"(MAE={best_mae['MAE_GLOBAL']:.3f})"
)

print(
    f"Meilleur RMSE       : "
    f"{best_rmse['MODELE']} "
    f"(RMSE={best_rmse['RMSE_GLOBAL']:.3f})"
)

print(
    f"Meilleur R²         : "
    f"{best_r2['MODELE']} "
    f"(R²={best_r2['R2_GLOBAL']:.3f})"
)

print(
    f"Plus stable         : "
    f"{best_stable['MODELE']}"
)

print(
    f"\nMODELE RETENU       : "
    f"{best_global['MODELE']}"
)

print(
    f"MAE global          : "
    f"{best_global['MAE_GLOBAL']:.3f}"
)

print(
    f"RMSE global         : "
    f"{best_global['RMSE_GLOBAL']:.3f}"
)

print(
    f"R² global           : "
    f"{best_global['R2_GLOBAL']:.3f}"
)

print(
    f"R² std global       : "
    f"{best_global['R2_STD_GLOBAL']:.3f}"
)

print("==============================================")


# ------------------------------------------------------------
# 13. Export
# ------------------------------------------------------------

classement_final.to_csv(
    OUTPUT_DIR / "classement_final_modeles_etape03.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "\n✅ classement_final_modeles_etape03.csv exporté"
)

,MODELE,MAE_MOYEN_125,MAE_MOYEN_105,MAE_GLOBAL,RMSE_GLOBAL,R2_MOYEN_125,R2_MOYEN_105,R2_GLOBAL,R2_STD_GLOBAL,ECART_MAE_COHORTES,ECART_R2_COHORTES,RANG_PERFORMANCE,RANG_STABILITE,SCORE_GLOBAL,RANG_GLOBAL
0,RandomForest_OPT,1.654217e+01,16.160341,1.635126e+01,2.025277e+01,2.139515e-01,0.273127,2.435393e-01,1.222639e-01,3.818330e-01,5.917556e-02,1.333333,5.50,3.0,1.0
1,HistGradientBoosting,1.647070e+01,16.335504,1.640310e+01,2.039272e+01,1.840970e-01,0.283063,2.335801e-01,1.257265e-01,1.351945e-01,9.896630e-02,2.333333,5.00,3.4,2.0
2,GradientBoosting_OPT,1.735218e+01,16.648406,1.700029e+01,2.058815e+01,1.926786e-01,0.247466,2.200725e-01,1.047750e-01,7.037750e-01,5.478791e-02,4.333333,5.00,4.6,3.0
3,RandomForest,1.706700e+01,16.219350,1.664318e+01,2.046196e+01,1.619270e-01,0.233250,1.975886e-01,1.770019e-01,8.476506e-01,7.132306e-02,3.666667,7.75,5.3,4.0
4,CatBoost,1.767492e+01,16.690035,1.718248e+01,2.092257e+01,1.518518e-01,0.232881,1.923666e-01,1.133554e-01,9.848897e-01,8.102972e-02,5.666667,6.00,5.8,5.0
5,ExtraTrees,1.778700e+01,17.485448,1.763623e+01,2.117214e+01,1.568726e-01,0.196850,1.768611e-01,1.153245e-01,3.015568e-01,3.997694e-02,7.000000,4.75,6.1,6.0
6,DecisionTree_OPT,1.705360e+01,15.564897,1.630925e+01,2.105089e+01,9.042579e-02,0.247878,1.691517e-01,2.084163e-01,1.488699e+00,1.574517e-01,4.666667,10.75,7.1,7.0
7,SVR_RBF,1.788214e+01,18.176985,1.802956e+01,2.204138e+01,1.324244e-01,0.084615,1.085198e-01,1.137139e-01,2.948490e-01,4.780906e-02,10.333333,4.00,7.8,8.0
8,Lasso,1.798928e+01,17.665365,1.782732e+01,2.185525e+01,8.603899e-02,0.081576,8.380747e-02,2.428483e-01,3.239186e-01,4.463048e-03,9.333333,7.25,8.5,9.0
9,DecisionTree,1.764204e+01,15.880854,1.676145e+01,2.138550e+01,3.455227e-02,0.192167,1.133597e-01,2.355703e-01,1.761186e+00,1.576149e-01,7.000000,12.25,9.1,10.0



        SYNTHÈSE FINALE ÉTAPE 03
Meilleur MAE        : DecisionTree_OPT (MAE=16.309)
Meilleur RMSE       : RandomForest_OPT (RMSE=20.253)
Meilleur R²         : RandomForest_OPT (R²=0.244)
Plus stable         : SVR_RBF

MODELE RETENU       : RandomForest_OPT
MAE global          : 16.351
RMSE global         : 20.253
R² global           : 0.244
R² std global       : 0.122

✅ classement_final_modeles_etape03.csv exporté


In [73]:
# ============================================================
# DERNIÈRE CELLULE  — Export passerelle vers notebook 04
# ============================================================

import json

model_base_125.to_csv(
    OUTPUT_DIR / "base_modelisation_MT1_N125.csv",
    index=False,
    encoding="utf-8-sig"
)

model_base_105.to_csv(
    OUTPUT_DIR / "base_modelisation_MT1_N105.csv",
    index=False,
    encoding="utf-8-sig"
)

configuration_features = {
    "FEATURES_MT1_NUM": FEATURES_MT1_NUM,
    "FEATURES_MT1_CAT": FEATURES_MT1_CAT,
    "TARGET": "TARGET_T2",
    "FRAMEWORK": "M_T1",
    "MODELE_RETENU": "RandomForest"
}

with open(
    OUTPUT_DIR / "configuration_MT1.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        configuration_features,
        f,
        ensure_ascii=False,
        indent=4
    )

print("✅ Passerelle 03 → 04 créée")
print("Base N=125 :", model_base_125.shape)
print("Base N=105 :", model_base_105.shape)
print("Modèle retenu : M_T1 + RandomForest")

✅ Passerelle 03 → 04 créée
Base N=125 : (125, 52)
Base N=105 : (105, 52)
Modèle retenu : M_T1 + RandomForest


In [74]:
# ============================================================
# CELLULE 51 — Test de variantes simples (version autonome)
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import RepeatedKFold, cross_validate
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

base125 = pd.read_csv(OUTPUT_DIR / "base_modelisation_MT1_N125.csv")
base105 = pd.read_csv(OUTPUT_DIR / "base_modelisation_MT1_N105.csv")
print("N=125 :", base125.shape, "| N=105 :", base105.shape)

# --- 2. Préprocesseur (même logique que votre cellule 25) ---
def creer_preprocesseur(variables_num, variables_cat):
    transformers = []
    if variables_num:
        transformers.append((
            "num",
            Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]),
            variables_num
        ))
    if variables_cat:
        transformers.append((
            "cat",
            Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore",
                                               drop="first",
                                               sparse_output=False))]),
            variables_cat
        ))
    return ColumnTransformer(transformers, remainder="drop")

# --- 3. Nouvelle variable : T0 du propre parcours ---
def ajouter_t0_propre(df):
    df = df.copy()
    df["T0__S_PROPRE"] = np.select(
        [df["T1__PARCOURS"] == "DA", df["T1__PARCOURS"] == "BI",
         df["T1__PARCOURS"] == "DS", df["T1__PARCOURS"] == "IA"],
        [df["T0__S_DA"], df["T0__S_BI"], df["T0__S_DS"], df["T0__S_IA"]],
        default=np.nan
    )
    return df

base125 = ajouter_t0_propre(base125)
base105 = ajouter_t0_propre(base105)

NIVEAUX = ["P__NIVEAU_INFORMATIQUE", "P__NIVEAU_EXCEL", "P__NIVEAU_POWER_BI",
           "P__NIVEAU_PYTHON", "P__NIVEAU_IA"]

JEUX = {
    "A. T1 seul":                   (["T1__SCORE"], []),
    "B. T1 + parcours":             (["T1__SCORE"], ["T1__PARCOURS"]),
    "C. T1 + T0 propre + parcours": (["T1__SCORE", "T0__S_PROPRE"], ["T1__PARCOURS"]),
    "D. C + niveaux auto-déclarés": (["T1__SCORE", "T0__S_PROPRE"] + NIVEAUX, ["T1__PARCOURS"]),
}

MODELES_TEST = {
    "Ridge(10)": Ridge(alpha=10),
    "RandomForest": RandomForestRegressor(
        n_estimators=500, min_samples_leaf=5, max_features=0.5,
        random_state=42, n_jobs=-1),
}

cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

# --- 4. Évaluation ---
lignes = []
for nom_base, base in [("N=125", base125), ("N=105", base105)]:
    y = base["TARGET_T2"].astype(float)
    for nom_jeu, (num, cat) in JEUX.items():
        X = base[num + cat]
        for nom_mod, mod in MODELES_TEST.items():
            pipe = Pipeline([("prep", creer_preprocesseur(num, cat)),
                             ("model", mod)])
            r = cross_validate(
                pipe, X, y, cv=cv,
                scoring=["neg_mean_absolute_error",
                         "neg_root_mean_squared_error", "r2"]
            )
            lignes.append({
                "COHORTE": nom_base, "JEU": nom_jeu, "MODELE": nom_mod,
                "MAE": -r["test_neg_mean_absolute_error"].mean(),
                "RMSE": -r["test_neg_root_mean_squared_error"].mean(),
                "R2": r["test_r2"].mean(),
                "R2_STD": r["test_r2"].std()
            })

resultats_variantes = pd.DataFrame(lignes).sort_values(["COHORTE", "MAE"])
display(resultats_variantes.round(3))


# --- 5. Export ---
resultats_variantes.to_csv(
    OUTPUT_DIR
    / "comparaison_variantes_T1_seul.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "✅ Comparaison T1 seul exportée"
)

N=125 : (125, 52) | N=105 : (105, 52)


,COHORTE,JEU,MODELE,MAE,RMSE,R2,R2_STD
9,N=105,A. T1 seul,RandomForest,16.866,20.576,0.189,0.143
15,N=105,D. C + niveaux auto-déclarés,RandomForest,17.449,21.086,0.157,0.128
11,N=105,B. T1 + parcours,RandomForest,17.835,21.402,0.127,0.136
13,N=105,C. T1 + T0 propre + parcours,RandomForest,17.875,21.329,0.135,0.122
8,N=105,A. T1 seul,Ridge(10),17.964,21.781,0.099,0.131
10,N=105,B. T1 + parcours,Ridge(10),18.289,22.185,0.066,0.138
12,N=105,C. T1 + T0 propre + parcours,Ridge(10),18.290,22.256,0.059,0.140
14,N=105,D. C + niveaux auto-déclarés,Ridge(10),18.481,22.650,0.025,0.165
7,N=125,D. C + niveaux auto-déclarés,RandomForest,17.413,21.207,0.151,0.129
1,N=125,A. T1 seul,RandomForest,17.504,21.439,0.124,0.173


✅ Comparaison T1 seul exportée
